# SurfaceGuard AI — Kaggle Master V9.2.5 (Five-Model, Pooled-Metric, Strict Submission)

**Purpose:** run the V9.2 five-model calibration and inference workflow in Kaggle Notebooks. This is the Kaggle-native counterpart of the repaired Colab V9.2.4 workflow, adapted using the uploaded Kaggle V15 notebook for input discovery, GPU safety, and `/kaggle/working` submission handling.

## Five models retained

1. V4 — U-Net / ResNet34
2. V5 — SegFormer-B0
3. V6 — DeepLabV3+ / ResNet50
4. MiT-B4 — SegFormer-B4
5. Scratch — U-Net / ResNet34

The notebook keeps the **five-model joint calibration search, class-aware blending, pooled official metric, fixed reserved-holdout evaluation, and the null-safe RLE submission validator**. It does **not** replace the workflow with the four-model V14/V15 inference-only fallback or the separate eight-model V15.2 hybrid.

## Kaggle inputs required

Attach these under **Notebook → Add Input**:

1. Competition: `industrial-surface-defect-detection`.
2. Checkpoint dataset: `shrutikarthikeyan/v4-v5-v6-mit` (contains the V4/V5/V6/MiT-B4 checkpoints).
3. If available, attach the Kaggle dataset/ZIP containing `model_a_scratch.pt` from your Colab V7 outputs. The notebook can reuse this as the fifth model after checking that its segmentation weights match. If no compatible scratch checkpoint is attached, the notebook trains the scratch U-Net once; a GPU is required for that path.

Kaggle input datasets are read-only. The notebook writes checkpoints, caches, reports, and the final CSV under `/kaggle/working` or `/kaggle/temp`.

## Runtime settings

- Enable **GPU** in Kaggle Notebook Settings. CUDA is selected only after a successful initialization and free-memory check; otherwise CPU mode is selected. A later CUDA out-of-memory error stops the run rather than silently changing calibration/test inference modes.
- Enable **Internet** because the SegFormer configurations are loaded from Hugging Face model IDs.
- Keep the competition rules accepted in your Kaggle account.
- The notebook uses Kaggle's attached inputs; it does not mount Google Drive and does not need a Kaggle API token.

## Submission output

The final file is `/kaggle/working/submission.csv`. It is independently reloaded and checked against the official template. Genuine empty masks are exactly one space (`" "`); nulls, missing predictions, duplicate keys, invalid RLE, incorrect row counts, and ordering mismatches raise errors.

**No leaderboard result is guaranteed.** The previous 0.725678 value was a local holdout result; the best confirmed public score remains 0.63881. Keep the V13 submission as your fallback until Kaggle processes this candidate.


In [1]:
# Kaggle environment check
from pathlib import Path

print("Running inside Kaggle:", Path("/kaggle/input").exists())
print("Kaggle input root exists:", Path("/kaggle/input").exists())
print("Kaggle working root exists:", Path("/kaggle/working").exists())
print("CUDA visible before setup:", __import__("torch").cuda.is_available())


Running inside Kaggle: True
Kaggle input root exists: True
Kaggle working root exists: True
CUDA visible before setup: True


## Setup

In [2]:
# Match the supplied Kaggle V15 environment setup: do not replace Kaggle's PyTorch/CUDA wheel.
%pip install -q --disable-pip-version-check --no-deps segmentation-models-pytorch==0.6.0

from importlib.metadata import version
import torch
import segmentation_models_pytorch as smp

print("Installed SMP version:", version("segmentation-models-pytorch"))
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Note: you may need to restart the kernel to use updated packages.
Installed SMP version: 0.6.0
PyTorch: 2.11.0+cu130
PyTorch CUDA build: 13.0
CUDA available: True
GPU: Tesla T4


In [3]:
import os
import csv
import json
import math
import time
import random
import shutil
import zipfile
import hashlib
import copy
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
from transformers import SegformerConfig, SegformerForSemanticSegmentation
import re

In [4]:
import os
import gc
import shutil
from pathlib import Path, PurePosixPath

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
TEMP_ROOT = Path("/kaggle/temp")

if not INPUT_ROOT.exists():
    raise FileNotFoundError(
        "/kaggle/input is unavailable. Open this notebook in Kaggle and attach "
        "the competition and checkpoint datasets using Add Input."
    )
WORK_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
N_CLASSES = len(CLASSES)
HEIGHT, WIDTH = 128, 800
DEVICE = torch.device("cpu")
GPU_NAME = "CPU fallback"
GPU_FREE_GB = 0.0
GPU_TOTAL_GB = 0.0

# Select CUDA only if the device initialises and has adequate free memory.
MIN_FREE_GPU_GB = 5.0
if torch.cuda.is_available():
    try:
        torch.cuda.init()
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        GPU_FREE_GB = free_bytes / (1024 ** 3)
        GPU_TOTAL_GB = total_bytes / (1024 ** 3)
        if GPU_FREE_GB >= MIN_FREE_GPU_GB:
            _probe = torch.empty((1,), device="cuda:0")
            torch.cuda.synchronize()
            del _probe
            DEVICE = torch.device("cuda:0")
            GPU_NAME = torch.cuda.get_device_name(0)
            torch.cuda.manual_seed_all(SEED)
            torch.backends.cudnn.benchmark = False
        else:
            print(
                f"CUDA is visible but only {GPU_FREE_GB:.2f} GiB is free; "
                f"the {MIN_FREE_GPU_GB:.1f} GiB safety threshold was not met. Using CPU."
            )
    except Exception as exc:
        DEVICE = torch.device("cpu")
        GPU_NAME = "CPU fallback (CUDA initialisation failed)"
        print("CUDA initialisation failed; using CPU:", type(exc).__name__, str(exc)[:250])

OUTPUT_DIR = WORK_ROOT / "surfaceguard_v9_2_5_outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CACHE_DIR = TEMP_ROOT / "surfaceguard_v9_2_5_cache"
MODEL_CACHE_DIR = TEMP_ROOT / "surfaceguard_v9_2_5_model_cache"
for directory in (OUTPUT_DIR, CHECKPOINT_DIR, CACHE_DIR, MODEL_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# All Kaggle-attached datasets are read-only under INPUT_ROOT.
# The checkpoint discovery cell searches this tree and safely extracts matching
# files from attached ZIPs. The competition data root is located separately.
CUSTOM_DIR = INPUT_ROOT
COMPETITION = "industrial-surface-defect-detection"

def locate_competition_root(root):
    root = Path(root)
    preferred = root / "competitions" / COMPETITION / "public"
    if (
        (preferred / "train.csv").is_file()
        and (preferred / "sample_submission.csv").is_file()
        and (preferred / "train").is_dir()
        and (preferred / "test").is_dir()
    ):
        return preferred

    candidates = []
    for train_csv in root.rglob("train.csv"):
        parent = train_csv.parent
        if (
            (parent / "sample_submission.csv").is_file()
            and (parent / "train").is_dir()
            and (parent / "test").is_dir()
        ):
            candidates.append(parent.resolve())

    candidates = sorted(set(candidates), key=str)
    if len(candidates) == 1:
        return Path(candidates[0])
    if len(candidates) > 1:
        preferred_matches = [p for p in candidates if COMPETITION in str(p)]
        if len(preferred_matches) == 1:
            return Path(preferred_matches[0])
        raise RuntimeError(
            "More than one possible competition data root was found. "
            "Keep the intended competition input attached:\n" + "\n".join(map(str, candidates))
        )
    return None

DATA_ROOT = locate_competition_root(INPUT_ROOT)
if DATA_ROOT is None:
    discovered_csvs = [str(p) for p in INPUT_ROOT.rglob("train.csv")][:20]
    raise FileNotFoundError(
        "Could not locate the competition files under /kaggle/input. Expected a folder "
        "containing train.csv, sample_submission.csv, train/, and test/. "
        "Attach the industrial-surface-defect-detection competition input. "
        f"Discovered train.csv files: {discovered_csvs}"
    )

TRAIN_IMAGE_DIR = DATA_ROOT / "train"
TEST_IMAGE_DIR = DATA_ROOT / "test"
TRAIN_CSV_PATH = DATA_ROOT / "train.csv"
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"

WANDB_ENTITY = "23f3003529-dl-genai-project"
WANDB_PROJECT = "23f3003529-t32026"
WANDB_HOST = "https://api.forge.coreweave.com"
WANDB_ENABLED = False
os.environ["WANDB_MODE"] = "disabled"

print("=" * 72)
print("SURFACEGUARD V9.2.5 — KAGGLE SETUP COMPLETE")
print("=" * 72)
print("Competition data root:", DATA_ROOT)
print("Attached Kaggle input roots:", sorted(p.name for p in INPUT_ROOT.iterdir()))
print("Checkpoint search root:", CUSTOM_DIR)
print("Output directory:", OUTPUT_DIR)
print("Checkpoint output directory:", CHECKPOINT_DIR)
print("Temporary cache:", CACHE_DIR)
print("Device:", DEVICE, "|", GPU_NAME)
if DEVICE.type == "cuda":
    print(f"GPU memory at startup: {GPU_FREE_GB:.2f}/{GPU_TOTAL_GB:.2f} GiB free/total")
else:
    print("CPU mode selected. The scratch model cannot be trained on CPU if its checkpoint is missing.")
print("W&B disabled; it is not required for model loading, calibration, or submission.")


SURFACEGUARD V9.2.5 — KAGGLE SETUP COMPLETE
Competition data root: /kaggle/input/competitions/industrial-surface-defect-detection/public
Attached Kaggle input roots: ['competitions', 'datasets']
Checkpoint search root: /kaggle/input
Output directory: /kaggle/working/surfaceguard_v9_2_5_outputs
Checkpoint output directory: /kaggle/working/surfaceguard_v9_2_5_outputs/checkpoints
Temporary cache: /kaggle/temp/surfaceguard_v9_2_5_cache
Device: cuda:0 | Tesla T4
GPU memory at startup: 14.46/14.56 GiB free/total
W&B disabled; it is not required for model loading, calibration, or submission.


In [5]:
USE_TTA_AT_FINAL_INFERENCE = DEVICE.type == "cuda"
USE_TTA_FOR_CALIBRATION = USE_TTA_AT_FINAL_INFERENCE
if USE_TTA_FOR_CALIBRATION:
    print("GPU detected. Four-way flip TTA will be used consistently for calibration, holdout, and final test inference.")
else:
    print("CPU fallback: single-pass inference will be used consistently for calibration, holdout, and test. Expect a much longer run.")

GPU detected. Four-way flip TTA will be used consistently for calibration, holdout, and final test inference.


In [6]:
required_paths = [TRAIN_CSV_PATH, SAMPLE_SUBMISSION_PATH, TRAIN_IMAGE_DIR, TEST_IMAGE_DIR]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Competition data preflight failed:\n- " + "\n- ".join(missing_paths))

train_image_files = list(TRAIN_IMAGE_DIR.glob("*.jpg"))
test_image_files = list(TEST_IMAGE_DIR.glob("*.jpg"))
if not train_image_files or not test_image_files:
    raise FileNotFoundError("Train/test folders exist but contain no .jpg files.")

train_header = pd.read_csv(TRAIN_CSV_PATH, nrows=5)
sample_header = pd.read_csv(SAMPLE_SUBMISSION_PATH, nrows=5)
if not {"image_id", "defect_type", "mask_rle"}.issubset(train_header.columns):
    raise ValueError("train.csv does not contain image_id, defect_type, mask_rle.")
if sample_header.columns.tolist() != ["ID", "image_id", "defect_type", "mask_rle"]:
    raise ValueError(f"Unexpected sample_submission columns: {sample_header.columns.tolist()}")

print("Competition data preflight: PASSED")
print("Training JPGs:", len(train_image_files), "| Test JPGs:", len(test_image_files))
print("Train CSV:", TRAIN_CSV_PATH)
print("Sample submission:", SAMPLE_SUBMISSION_PATH)

Competition data preflight: PASSED
Training JPGs: 11393 | Test JPGs: 3000
Train CSV: /kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv
Sample submission: /kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv


## Kaggle input discovery and checkpoint recovery

The competition data and model checkpoints are read from Kaggle's attached `/kaggle/input` datasets. This notebook does **not** download files through the Kaggle API, does not need `kaggle.json`, and does not mount Google Drive.

Attach `shrutikarthikeyan/v4-v5-v6-mit` for the four established segmentation checkpoints. If your attached Colab V7 checkpoint dataset/ZIP also contains `model_a_scratch.pt`, Cell 11 can recover it as the fifth model. Otherwise, the notebook trains the scratch U-Net once on a usable GPU and saves the checkpoint under `/kaggle/working/surfaceguard_v9_2_5_outputs/checkpoints/`.


In [7]:
print("Detected competition directory:", DATA_ROOT)
print("Competition files:", sorted(p.name for p in DATA_ROOT.iterdir())[:20])
print("Attached checkpoint ZIP archives:", [p.name for p in CUSTOM_DIR.glob("*.zip")])
print("Existing scratch checkpoint:", (CHECKPOINT_DIR / "model_scratch.pt").is_file())

Detected competition directory: /kaggle/input/competitions/industrial-surface-defect-detection/public
Competition files: ['sample_submission.csv', 'test', 'train', 'train.csv']
Attached checkpoint ZIP archives: []
Existing scratch checkpoint: False


In [8]:
# W&B is optional and intentionally disabled in this Kaggle submission notebook.
# It is not required for model training, calibration, inference, or file generation.
WANDB_ENABLED = False
os.environ["WANDB_MODE"] = "disabled"
print("W&B logging disabled; continuing without external experiment tracking.")


W&B logging disabled; continuing without external experiment tracking.


In [9]:
# Extract direct checkpoint files and reconstruct the split PyTorch ZIP archives for V4/V5.
import hashlib

MODEL_CACHE_DIR = Path("/kaggle/temp/surfaceguard_v9_2_5_model_cache")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED_ARTIFACTS = {
    "best_v6_deeplabv3plus.pt", "best_mit_b4.pt", "best_v4.pt", "best_v5.pt",
    "model_scratch.pt", "model_a_scratch.pt"
}


def _safe_zip_parts(member_name):
    member = PurePosixPath(member_name)
    if member.is_absolute() or ".." in member.parts:
        return None
    return member.parts


def _copy_zip_member(zf, member, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zf.open(member, "r") as source, open(destination, "wb") as target:
        shutil.copyfileobj(source, target)


extracted_checkpoint_members = []
for zip_path in sorted(CUSTOM_DIR.rglob("*.zip")):
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            identity = hashlib.sha256(str(zip_path.resolve()).encode("utf-8")).hexdigest()[:12]
            stage_root = MODEL_CACHE_DIR / "zip_extracted" / f"{zip_path.stem}_{identity}"
            for info in zf.infolist():
                if info.is_dir():
                    continue
                parts = _safe_zip_parts(info.filename)
                if not parts:
                    continue
                basename = parts[-1]
                if basename in REQUIRED_ARTIFACTS:
                    target = stage_root / "artifacts" / Path(*parts)
                    if not target.exists():
                        _copy_zip_member(zf, info, target)
                        extracted_checkpoint_members.append(target)
                    continue
                for model_name in ("best_v4", "best_v5"):
                    matching = [i for i, part in enumerate(parts) if part == model_name]
                    if matching:
                        root_index = matching[-1]
                        relative_parts = parts[root_index:]
                        if len(relative_parts) >= 2:
                            target = stage_root / "archives" / Path(*relative_parts)
                            if not target.exists():
                                _copy_zip_member(zf, info, target)
                                extracted_checkpoint_members.append(target)
                        break
    except zipfile.BadZipFile:
        print("Skipping damaged/non-ZIP file:", zip_path)


def _sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_unique_checkpoint(filename, required=True):
    hits = []
    for root in (CUSTOM_DIR, MODEL_CACHE_DIR):
        if root.exists():
            hits.extend(p for p in root.rglob(filename) if p.is_file())
    hits = list({str(p.resolve()): p for p in hits}.values())
    if not hits:
        if required:
            raise FileNotFoundError(f"Required checkpoint {filename} was not found in {CUSTOM_DIR} or {MODEL_CACHE_DIR}.")
        return None
    if len(hits) == 1:
        return hits[0]
    if len({p.stat().st_size for p in hits}) == 1 and len({_sha256(p) for p in hits}) == 1:
        return sorted(hits, key=lambda p: (len(str(p)), str(p)))[0]
    listing = "\n".join(f"{p} ({p.stat().st_size:,} bytes)" for p in hits)
    raise RuntimeError(f"Conflicting duplicate checkpoint files for {filename}:\n{listing}")


def reconstruct_split_checkpoint(model_name):
    direct = find_unique_checkpoint(f"{model_name}.pt", required=False)
    if direct is not None:
        return direct
    roots = []
    for root in (CUSTOM_DIR, MODEL_CACHE_DIR):
        if root.exists():
            for data_pickle in root.rglob("data.pkl"):
                archive_root = data_pickle.parent
                if archive_root.name == model_name and (archive_root / "version").is_file():
                    roots.append(archive_root)
    roots = list({str(p.resolve()): p for p in roots}.values())
    if len(roots) != 1:
        raise FileNotFoundError(
            f"Expected one split PyTorch archive for {model_name}; found {roots}. "
            "The checkpoint dataset should contain best_v4/data.pkl and best_v5/data.pkl."
        )
    output_path = MODEL_CACHE_DIR / f"{model_name}.pt"
    with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_STORED) as zf:
        for file_path in roots[0].rglob("*"):
            if file_path.is_file():
                archive_name = (Path(model_name) / file_path.relative_to(roots[0])).as_posix()
                zf.write(file_path, archive_name)
    if not zipfile.is_zipfile(output_path):
        raise RuntimeError(f"Reconstructed checkpoint archive is invalid: {output_path}")
    return output_path


CHECKPOINT_PATHS = {
    "best_v4.pt": reconstruct_split_checkpoint("best_v4"),
    "best_v5.pt": reconstruct_split_checkpoint("best_v5"),
    "best_v6_deeplabv3plus.pt": find_unique_checkpoint("best_v6_deeplabv3plus.pt"),
    "best_mit_b4.pt": find_unique_checkpoint("best_mit_b4.pt"),
}

# Prefer the canonical V9 scratch checkpoint. If it is absent, use the legacy
# Colab V7 Model A scratch U-Net checkpoint; Cell 35 validates shared segmentation
# weights and permits only its unused auxiliary classification-head keys.
def _unique_scratch_hit(filename):
    hits = []
    for root in (CUSTOM_DIR, MODEL_CACHE_DIR, OUTPUT_DIR):
        if root.exists():
            hits.extend(p for p in root.rglob(filename) if p.is_file())
    hits = list({str(p.resolve()): p for p in hits}.values())
    if not hits:
        return None
    if len(hits) == 1:
        return hits[0]
    hashes = {_sha256(p) for p in hits}
    if len(hashes) == 1:
        return sorted(hits, key=lambda p: (len(str(p)), str(p)))[0]
    listing = "\n".join(f"{p} ({p.stat().st_size:,} bytes)" for p in hits)
    raise RuntimeError(f"Conflicting copies of {filename} were found:\n{listing}")

existing_scratch = _unique_scratch_hit("model_scratch.pt")
scratch_source_name = "model_scratch.pt"
if existing_scratch is None:
    existing_scratch = _unique_scratch_hit("model_a_scratch.pt")
    scratch_source_name = "model_a_scratch.pt"

SCRATCH_CHECKPOINT_PATH = CHECKPOINT_DIR / "model_scratch.pt"
if existing_scratch is not None and existing_scratch.resolve() != SCRATCH_CHECKPOINT_PATH.resolve():
    shutil.copy2(existing_scratch, SCRATCH_CHECKPOINT_PATH)

print("\nCHECKPOINT PREFLIGHT: PASSED")
for filename, path in CHECKPOINT_PATHS.items():
    print(f"{filename:32s} {path} ({Path(path).stat().st_size / 1024**2:.1f} MiB)")
print("Scratch checkpoint source:", str(existing_scratch) if existing_scratch is not None else "not found in attached Kaggle inputs or working output")
print("Scratch checkpoint source filename:", scratch_source_name if existing_scratch is not None else "none")
print("Scratch checkpoint destination:", SCRATCH_CHECKPOINT_PATH)
print("Scratch checkpoint:", "found — segmentation compatibility checked in Cell 35" if SCRATCH_CHECKPOINT_PATH.is_file() else "not found — Cell 35 will train it once if CUDA is available")


CHECKPOINT PREFLIGHT: PASSED
best_v4.pt                       /kaggle/input/datasets/shrutikarthikeyan/v4-v5-v6-mit/SurfaceGuard_V4_V5_Ensemble_output/reconstructed_checkpoints/best_v4.pt (279.8 MiB)
best_v5.pt                       /kaggle/input/datasets/shrutikarthikeyan/v4-v5-v6-mit/SurfaceGuard_V4_V5_Ensemble_output/reconstructed_checkpoints/best_v5.pt (42.7 MiB)
best_v6_deeplabv3plus.pt         /kaggle/input/datasets/shrutikarthikeyan/v4-v5-v6-mit/SurfaceGuard_V4_V5_Ensemble_output/surfaceguard_v6_deeplabv3plus/best_v6_deeplabv3plus.pt (305.8 MiB)
best_mit_b4.pt                   /kaggle/input/datasets/shrutikarthikeyan/v4-v5-v6-mit/SurfaceGuard_MiT_B4_Aug_TTA/surfaceguard_mit_b4_experiment/best_mit_b4.pt (733.5 MiB)
Scratch checkpoint source: /kaggle/input/datasets/shrutikarthikeyan/surfaceguard-colab-v7-checkpoints/model_a_scratch.pt
Scratch checkpoint source filename: model_a_scratch.pt
Scratch checkpoint destination: /kaggle/working/surfaceguard_v9_2_5_outputs/checkpoints/mod

## Data pipeline and the three-way split

Train / calibration / holdout, not train / val. Calibration is where blend weights and thresholds get searched; holdout is touched exactly once, at the end, for honest reporting — this is the structure your own notebook already proved out, kept as-is here.

In [10]:
def decode_rle(rle_string, shape):
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if isinstance(rle_string, str) and rle_string.strip() not in ("", "-1"):
        tokens = rle_string.split()
        starts = np.asarray(tokens[0::2], dtype=np.int64) - 1
        lengths = np.asarray(tokens[1::2], dtype=np.int64)
        for start, length in zip(starts, lengths):
            mask[start:start + length] = 1
    return mask.reshape(shape, order="F")


def encode_rle(mask):
    pixels = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    if not np.any(pixels):
        # Match the successfully executed V15 notebook: an empty mask is one
        # literal space, never NaN and never a zero-length CSV field.
        return " "
    padded = np.concatenate([[0], pixels, [0]])
    changes = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    starts = changes[0::2]
    lengths = changes[1::2] - starts
    encoded = np.empty(starts.size * 2, dtype=np.int64)
    encoded[0::2] = starts
    encoded[1::2] = lengths
    return " ".join(map(str, encoded.tolist()))

In [11]:
_self_test_mask = np.zeros((HEIGHT, WIDTH), dtype=np.uint8)
_self_test_mask[10:15, 20:25] = 1
_self_test_mask[100:103, 700:720] = 1
_rle = encode_rle(_self_test_mask)
_decoded = decode_rle(_rle, (HEIGHT, WIDTH))
assert np.array_equal(_self_test_mask, _decoded), "RLE round-trip failed"
print("RLE round-trip OK")

# Empty-mask convention test: exactly one space, decodes to an all-zero mask.
_empty_rle_test_mask = np.zeros((HEIGHT, WIDTH), dtype=np.uint8)
_empty_rle_text = encode_rle(_empty_rle_test_mask)
assert _empty_rle_text == " ", f"Empty-mask RLE must be one space, got {_empty_rle_text!r}"
assert np.array_equal(
    decode_rle(_empty_rle_text, (HEIGHT, WIDTH)), _empty_rle_test_mask
), "Empty-mask RLE round-trip failed"
print("Empty-mask single-space RLE self-test: PASSED")


RLE round-trip OK
Empty-mask single-space RLE self-test: PASSED


In [12]:
train_df_raw = pd.read_csv(
    TRAIN_CSV_PATH,
    dtype={"image_id": str, "defect_type": str}
)
sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH,
    dtype={"ID": str, "image_id": str, "defect_type": str}
)
image_ids = sorted(train_df_raw["image_id"].astype(str).unique())
print("Unique training images:", len(image_ids))
if len(image_ids) != 11393:
    raise ValueError(f"Expected 11,393 unique training images; found {len(image_ids)}")

Unique training images: 11393


In [13]:
def build_mask_lookup(dataframe, image_ids, classes, height, width):
    lookup = {}
    grouped = dataframe.groupby("image_id")
    for image_id in tqdm(image_ids, desc="Decoding masks"):
        stack = np.zeros((len(classes), height, width), dtype=np.uint8)
        if image_id in grouped.groups:
            rows = grouped.get_group(image_id)
            for _, row in rows.iterrows():
                if row["defect_type"] in classes:
                    class_idx = classes.index(row["defect_type"])
                    stack[class_idx] = decode_rle(row["mask_rle"], (height, width))
        lookup[image_id] = stack
    return lookup


mask_lookup = build_mask_lookup(train_df_raw, image_ids, CLASSES, HEIGHT, WIDTH)

Decoding masks:   0%|          | 0/11393 [00:00<?, ?it/s]

In [14]:
# Recreate the exact outer 80/20 split used by the V13/V14 saved checkpoints.
# The stratification labels are the four binary class-presence bits in class order.
labels_for_split = train_df_raw[["image_id", "defect_type", "mask_rle"]].copy()
labels_for_split["image_id"] = labels_for_split["image_id"].astype(str)
labels_for_split["has_defect"] = labels_for_split["mask_rle"].fillna("").astype(str).str.strip().ne("")
presence = labels_for_split.pivot_table(
    index="image_id", columns="defect_type", values="has_defect",
    aggfunc="max", fill_value=False
).reindex(columns=CLASSES, fill_value=False).astype(bool)
all_image_ids = np.asarray(sorted(presence.index.astype(str)), dtype=str)
presence = presence.reindex(all_image_ids)
label_combo = presence.astype(int).astype(str).agg("".join, axis=1)

train_ids, val_ids = train_test_split(
    all_image_ids,
    test_size=0.20,
    random_state=SEED,
    stratify=label_combo.loc[all_image_ids]
)
train_ids = np.asarray(train_ids, dtype=str)
val_ids = np.asarray(val_ids, dtype=str)

if len(all_image_ids) != 11393 or len(train_ids) != 9114 or len(val_ids) != 2279:
    raise RuntimeError(
        f"Unexpected seed-42 outer split: total={len(all_image_ids)}, train={len(train_ids)}, val={len(val_ids)}. "
        "Check that this is the expected competition dataset version."
    )
if set(train_ids).intersection(set(val_ids)):
    raise RuntimeError("Training/validation image overlap detected.")

labels_df = pd.DataFrame({"image_id": all_image_ids, "label_combo": label_combo.loc[all_image_ids].to_numpy()})
train_df = labels_df[labels_df["image_id"].isin(set(train_ids))].reset_index(drop=True)
val_df = labels_df[labels_df["image_id"].isin(set(val_ids))].reset_index(drop=True)

# Split the original unseen 20% into calibration and reserved holdout.
try:
    cal_df, holdout_df = train_test_split(
        val_df, test_size=0.40, random_state=2026, stratify=val_df["label_combo"]
    )
    INTERNAL_SPLIT_NOTE = "stratified by the exact four-bit class-presence combination"
except ValueError as exc:
    warnings.warn(f"Internal calibration/holdout stratification was not possible: {exc}. Using deterministic fallback.")
    cal_df, holdout_df = train_test_split(val_df, test_size=0.40, random_state=2026)
    INTERNAL_SPLIT_NOTE = "deterministic random fallback"

train_df = train_df.reset_index(drop=True)
cal_df = cal_df.reset_index(drop=True)
holdout_df = holdout_df.reset_index(drop=True)

if set(train_df["image_id"]).intersection(set(cal_df["image_id"])) or set(train_df["image_id"]).intersection(set(holdout_df["image_id"])):
    raise RuntimeError("An image appears in both training and the unseen validation partitions.")
if set(cal_df["image_id"]).intersection(set(holdout_df["image_id"])):
    raise RuntimeError("Calibration/holdout image overlap detected.")

print("Train:", len(train_df), "| Calibration:", len(cal_df), "| Holdout:", len(holdout_df))
print("Outer split: exact V13/V14 seed-42 80/20 stratification")
print("Internal split:", INTERNAL_SPLIT_NOTE)

Train: 9114 | Calibration: 1367 | Holdout: 912
Outer split: exact V13/V14 seed-42 80/20 stratification
Internal split: stratified by the exact four-bit class-presence combination


## Dataset and augmentation

Flips, shift, brightness/contrast, noise, and cutout for training. Evaluation/test datasets apply none of this — same per-image z-score normalization throughout.

In [15]:
def random_shift(image, mask, max_shift_frac=0.04):
    height, width = image.shape
    shift_y = random.randint(-int(height * max_shift_frac), int(height * max_shift_frac))
    shift_x = random.randint(-int(width * max_shift_frac), int(width * max_shift_frac))
    if shift_y == 0 and shift_x == 0:
        return image, mask
    image = np.roll(image, shift=(shift_y, shift_x), axis=(0, 1))
    mask = np.roll(mask, shift=(shift_y, shift_x), axis=(1, 2))
    return image, mask


def cutout(image, max_patches=2, patch_frac=0.08):
    height, width = image.shape
    image = image.copy()
    for _ in range(random.randint(1, max_patches)):
        patch_h = max(1, int(height * patch_frac * random.uniform(0.5, 1.5)))
        patch_w = max(1, int(width * patch_frac * random.uniform(0.5, 1.5)))
        top = random.randint(0, max(height - patch_h, 0))
        left = random.randint(0, max(width - patch_w, 0))
        image[top:top + patch_h, left:left + patch_w] = float(image.mean())
    return image


def load_and_normalize(image_path, height, width):
    image = Image.open(image_path).convert("L")
    image = image.resize((width, height), Image.Resampling.BILINEAR)
    image = np.asarray(image, dtype=np.float32) / 255.0
    return image


def normalize_inplace(image):
    mean = float(image.mean())
    std = max(float(image.std()), 0.05)
    return (image - mean) / std


class SurfaceDefectDataset(Dataset):
    def __init__(self, dataframe, image_dir, mask_lookup, classes, height, width, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.mask_lookup = mask_lookup
        self.classes = classes
        self.height = height
        self.width = width
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = str(self.df.iloc[idx]["image_id"])
        image = load_and_normalize(self.image_dir / f"{image_id}.jpg", self.height, self.width)
        mask = self.mask_lookup[image_id].astype(np.float32).copy()

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1)
                mask = np.flip(mask, axis=2)
            if random.random() < 0.5:
                image = np.flip(image, axis=0)
                mask = np.flip(mask, axis=1)
            if random.random() < 0.3:
                image, mask = random_shift(image, mask)
            if random.random() < 0.5:
                gain = random.uniform(0.85, 1.15)
                bias = random.uniform(-0.08, 0.08)
                image = np.clip(image * gain + bias, 0.0, 1.0)
            if random.random() < 0.3:
                noise = np.random.normal(0.0, 0.02, size=image.shape).astype(np.float32)
                image = np.clip(image + noise, 0.0, 1.0)
            if random.random() < 0.25:
                image = cutout(image)

        image = normalize_inplace(image)
        image_tensor = torch.from_numpy(np.ascontiguousarray(image)).unsqueeze(0).float()
        mask_tensor = torch.from_numpy(np.ascontiguousarray(mask)).float()
        return image_tensor, mask_tensor, image_id


class TestDefectDataset(Dataset):
    def __init__(self, image_ids, image_dir, height, width):
        self.image_ids = list(image_ids)
        self.image_dir = Path(image_dir)
        self.height = height
        self.width = width

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image = load_and_normalize(self.image_dir / f"{image_id}.jpg", self.height, self.width)
        image = normalize_inplace(image)
        image_tensor = torch.from_numpy(np.ascontiguousarray(image)).unsqueeze(0).float()
        return image_id, image_tensor

In [16]:
_sample_ds = SurfaceDefectDataset(train_df.head(4), TRAIN_IMAGE_DIR, mask_lookup, CLASSES, HEIGHT, WIDTH, augment=True)
_img, _msk, _iid = _sample_ds[0]
assert _img.shape == (1, HEIGHT, WIDTH), _img.shape
assert _msk.shape == (N_CLASSES, HEIGHT, WIDTH), _msk.shape
print("Dataset smoke test OK —", _img.shape, _msk.shape)
del _sample_ds, _img, _msk, _iid

Dataset smoke test OK — torch.Size([1, 128, 800]) torch.Size([4, 128, 800])


In [17]:
def build_weighted_sampler(dataframe, mask_lookup, classes, cap=3.0):
    image_ids = dataframe["image_id"].astype(str).tolist()
    counts = np.zeros(len(classes), dtype=np.int64)
    for image_id in image_ids:
        counts += mask_lookup[image_id].any(axis=(1, 2)).astype(np.int64)

    weights = []
    n = len(image_ids)
    for image_id in image_ids:
        present_idx = np.where(mask_lookup[image_id].any(axis=(1, 2)))[0]
        if len(present_idx) == 0:
            weights.append(1.0)
        else:
            weights.append(max(min(cap, np.sqrt(n / max(int(counts[c]), 1))) for c in present_idx))
    return torch.as_tensor(weights, dtype=torch.double), counts


train_sample_weights, train_class_counts = build_weighted_sampler(train_df, mask_lookup, CLASSES)
train_sampler = torch.utils.data.WeightedRandomSampler(train_sample_weights, len(train_sample_weights), replacement=True)
print("Train positive-image counts per class:", dict(zip(CLASSES, train_class_counts.tolist())))

Train positive-image counts per class: {'defect_a': 865, 'defect_b': 250, 'defect_c': 3280, 'defect_d': 1365}


In [18]:
EVAL_BATCH_SIZE = 1  # conservative for five models + four-way TTA on a Kaggle T4
EVAL_NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
TRAIN_NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0


def build_eval_loader(dataframe, batch_size=EVAL_BATCH_SIZE, num_workers=EVAL_NUM_WORKERS):
    dataset = SurfaceDefectDataset(dataframe, TRAIN_IMAGE_DIR, mask_lookup, CLASSES, HEIGHT, WIDTH, augment=False)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=(DEVICE.type == "cuda"))


def build_train_loader(batch_size=8 if DEVICE.type == "cuda" else 2, num_workers=TRAIN_NUM_WORKERS):
    dataset = SurfaceDefectDataset(train_df, TRAIN_IMAGE_DIR, mask_lookup, CLASSES, HEIGHT, WIDTH, augment=True)
    return DataLoader(dataset, batch_size=batch_size, sampler=train_sampler, num_workers=num_workers, pin_memory=(DEVICE.type == "cuda"), drop_last=True)


cal_loader = build_eval_loader(cal_df)
holdout_loader = build_eval_loader(holdout_df)
print("Evaluation batch size:", EVAL_BATCH_SIZE, "| workers:", EVAL_NUM_WORKERS)

Evaluation batch size: 1 | workers: 2


## Loss

Hybrid BCE + Focal Tversky, unchanged from what's already been validated to avoid the probability-calibration problem a pure Tversky loss caused earlier in this project.

In [19]:
def focal_tversky_loss(logits, targets, alpha=0.3, beta=0.7, gamma=0.75, eps=1e-6):
    probabilities = torch.sigmoid(logits.float())
    targets = targets.float()
    dims = (0, 2, 3)
    tp = (probabilities * targets).sum(dim=dims)
    fp = (probabilities * (1 - targets)).sum(dim=dims)
    fn = ((1 - probabilities) * targets).sum(dim=dims)
    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return ((1 - tversky).clamp_min(0) ** gamma).mean()


def segmentation_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits.float(), targets.float())
    return 0.5 * bce + 0.5 * focal_tversky_loss(logits, targets)

In [20]:
_dummy_logits = torch.randn(4, N_CLASSES, 16, 16)
_dummy_targets = (torch.rand(4, N_CLASSES, 16, 16) > 0.9).float()
_loss_value = segmentation_loss(_dummy_logits, _dummy_targets)
assert torch.isfinite(_loss_value)
print("Segmentation loss smoke test OK —", float(_loss_value))

Segmentation loss smoke test OK — 0.8073322772979736


## Competition metric — pooled, not class-macro

The notebook optimizes the weighted competition score as three pooled components across all image-class pairs: 70% mean foreground Dice over positive pairs, 20% one minus mean false-positive pixel fraction over empty pairs, and 10% empty-pair accuracy. For class-specific thresholds and weights, each class contributes its correctly proportioned additive share to the single pooled score. The four additive shares sum to the score; they are **not** averaged as four independent class scores.


In [21]:
THRESHOLD_GRID = np.round(np.arange(0.01, 0.951, 0.01), 3).astype(np.float32)


def evaluate_weighted_metric(forward_fn, loader, thresholds, classes, height, width, device):
    """Choose one threshold per class using the pooled competition metric.

    Official score = 0.70 * mean foreground Dice over positive image-class pairs
                   + 0.20 * (1 - mean FP pixel fraction over empty pairs)
                   + 0.10 * fraction of empty pairs predicted completely empty.

    Per-class threshold search remains separable. `weighted_score` in each result
    is that class's additive contribution to the one pooled score, not a standalone
    class-macro score. Summing the four contributions gives the official metric.
    """
    num_t, num_c = len(thresholds), len(classes)
    pixels_per_pair = height * width
    pos_dice_sum = np.zeros((num_t, num_c), dtype=np.float64)
    fp_pixels_sum = np.zeros((num_t, num_c), dtype=np.float64)
    empty_correct_sum = np.zeros((num_t, num_c), dtype=np.float64)
    pos_counts = np.zeros(num_c, dtype=np.int64)
    empty_counts = np.zeros(num_c, dtype=np.int64)
    threshold_tensor = torch.as_tensor(thresholds, device=device).view(-1, 1, 1, 1, 1)

    with torch.inference_mode():
        for images, masks, _ in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                logits = forward_fn(images)
            probabilities = torch.sigmoid(logits.float())
            targets = masks >= 0.5
            target_counts = targets.sum(dim=(-2, -1))
            positive = target_counts > 0
            empty = ~positive
            pos_counts += positive.sum(dim=0).cpu().numpy()
            empty_counts += empty.sum(dim=0).cpu().numpy()

            predictions = probabilities.unsqueeze(0) >= threshold_tensor
            predicted_counts = predictions.sum(dim=(-2, -1))
            intersections = (predictions & targets.unsqueeze(0)).sum(dim=(-2, -1))
            pair_dice = 2.0 * intersections.float() / (predicted_counts + target_counts.unsqueeze(0)).clamp_min(1)
            pos_dice_sum += (pair_dice * positive.unsqueeze(0)).sum(dim=1).cpu().numpy()
            fp_pixels_sum += (predicted_counts * empty.unsqueeze(0)).sum(dim=1).cpu().numpy()
            empty_correct_sum += ((predicted_counts == 0) & empty.unsqueeze(0)).sum(dim=1).cpu().numpy()

    total_positive = int(pos_counts.sum())
    total_empty = int(empty_counts.sum())
    if total_positive == 0 or total_empty == 0:
        raise ValueError(f"Metric needs positive and empty image-class pairs; got positive={total_positive}, empty={total_empty}.")
    missing_positive = [classes[c] for c in range(num_c) if pos_counts[c] == 0]
    missing_empty = [classes[c] for c in range(num_c) if empty_counts[c] == 0]
    if missing_positive or missing_empty:
        raise ValueError(f"Per-class threshold search needs both pair types. Missing positive={missing_positive}; missing empty={missing_empty}.")

    contributions = np.zeros((num_t, num_c), dtype=np.float64)
    for c in range(num_c):
        contributions[:, c] = (
            0.70 * pos_dice_sum[:, c] / total_positive
            + 0.20 * (empty_counts[c] * pixels_per_pair - fp_pixels_sum[:, c]) / (total_empty * pixels_per_pair)
            + 0.10 * empty_correct_sum[:, c] / total_empty
        )

    selected = contributions.argmax(axis=0)
    results = []
    for c, name in enumerate(classes):
        idx = int(selected[c])
        dice = float(pos_dice_sum[idx, c] / pos_counts[c])
        fp_score = float(1.0 - fp_pixels_sum[idx, c] / (empty_counts[c] * pixels_per_pair))
        empty_acc = float(empty_correct_sum[idx, c] / empty_counts[c])
        results.append({
            "class": name,
            "threshold": float(thresholds[idx]),
            "positive_pair_dice": dice,
            "false_positive_score": fp_score,
            "empty_accuracy": empty_acc,
            "class_local_score": float(0.70 * dice + 0.20 * fp_score + 0.10 * empty_acc),
            "weighted_score": float(contributions[idx, c]),
            "positive_pairs": int(pos_counts[c]),
            "empty_pairs": int(empty_counts[c]),
            "grid_edge": bool(idx == 0 or idx == num_t - 1),
        })
        if results[-1]["grid_edge"]:
            print(f"WARNING: {name} threshold saturated at grid edge — widen THRESHOLD_GRID")

    official_score = float(sum(row["weighted_score"] for row in results))
    return results, contributions


In [22]:
_dummy_forward = lambda x: torch.randn(x.shape[0], N_CLASSES, HEIGHT, WIDTH, device=x.device)

# Every dummy batch contains one completely empty image and one image with a
# small positive region for EACH class. This ensures both metric denominators
# are non-zero regardless of the random seed.
_dummy_loader = []
for _batch_idx in range(3):
    _dummy_images = torch.randn(2, 1, HEIGHT, WIDTH)
    _dummy_masks = torch.zeros(2, N_CLASSES, HEIGHT, WIDTH, dtype=torch.float32)
    for _class_idx in range(N_CLASSES):
        _top = 4 + _class_idx * 3
        _dummy_masks[1, _class_idx, _top:_top + 4, 20:28] = 1.0
    _dummy_loader.append((_dummy_images, _dummy_masks, [f"empty_{_batch_idx}", f"positive_{_batch_idx}"]))

_results, _ = evaluate_weighted_metric(
    _dummy_forward, _dummy_loader, THRESHOLD_GRID,
    CLASSES, HEIGHT, WIDTH, torch.device("cpu")
)
assert len(_results) == N_CLASSES
for _result in _results:
    assert _result["positive_pairs"] > 0, _result
    assert _result["empty_pairs"] > 0, _result
    assert np.isfinite(_result["weighted_score"]), _result
    assert 0.0 <= _result["weighted_score"] <= 1.0001, _result
print("Weighted metric smoke test OK — positive and empty pairs verified for all classes")
del _dummy_loader, _dummy_images, _dummy_masks, _results, _result, _class_idx, _top, _batch_idx


Weighted metric smoke test OK — positive and empty pairs verified for all classes


## EMA and training loop — used only for the new scratch model

In [23]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point:
                    self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
                else:
                    self.shadow[k] = v.detach().clone()

In [24]:
def run_validation(model, forward_fn, loader, device, state_dict=None, validation_forward_fn=None):
    original_state = None
    if state_dict is not None:
        original_state = copy.deepcopy(model.state_dict())
        model.load_state_dict(state_dict, strict=True)
    model.eval()
    selected_forward = validation_forward_fn or forward_fn
    results, contributions = evaluate_weighted_metric(
        lambda images: selected_forward(model, images), loader, THRESHOLD_GRID, CLASSES, HEIGHT, WIDTH, device
    )
    if original_state is not None:
        model.load_state_dict(original_state, strict=True)
    model.train()
    return results, contributions


def train_model(model, forward_fn, train_loader, cal_loader, optimizer, scheduler, epochs, device, checkpoint_path, run_name, use_ema=True, ema_decay=0.999, run_config=None, validation_forward_fn=None):
    if WANDB_ENABLED:
        wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=run_name, config=run_config or {}, reinit=True)
    model.to(device)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    ema = EMA(model, decay=ema_decay) if use_ema else None

    best_score, best_state = -1.0, None
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, batches = 0.0, 0
        for images, masks, _ in tqdm(train_loader, desc=f"{run_name} | epoch {epoch}/{epochs}"):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                logits = forward_fn(model, images)
                loss = segmentation_loss(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()
            if ema is not None:
                ema.update(model)
            running_loss += float(loss.item())
            batches += 1

        train_loss = running_loss / max(batches, 1)
        results, _ = run_validation(model, forward_fn, cal_loader, device, validation_forward_fn=validation_forward_fn)
        class_scores = {r["class"]: r["weighted_score"] for r in results}
        official_score = float(sum(class_scores.values()))
        log_payload = {"epoch": epoch, "train_loss": train_loss, "calibration_official_weighted_score": official_score}
        log_payload.update({f"{k}_official_score_contribution": v for k, v in class_scores.items()})
        if WANDB_ENABLED:
            wandb.log(log_payload)
        history.append(log_payload)
        print(f"{run_name} | epoch {epoch}/{epochs} | train_loss {train_loss:.5f} | pooled calibration score {official_score:.5f}")

        if official_score > best_score:
            best_score = official_score
            best_state = copy.deepcopy(model.state_dict())

    final_best_state, final_best_score = best_state, best_score
    if ema is not None:
        ema_results, _ = run_validation(model, forward_fn, cal_loader, device, state_dict=ema.shadow, validation_forward_fn=validation_forward_fn)
        ema_score = float(sum(r["weighted_score"] for r in ema_results))
        print(f"{run_name} | best raw {best_score:.5f} | EMA {ema_score:.5f}")
        if WANDB_ENABLED:
            wandb.log({"ema_calibration_official_weighted_score": ema_score})
        if ema_score > final_best_score:
            final_best_state, final_best_score = copy.deepcopy(ema.shadow), ema_score

    torch.save({
        "model_state_dict": final_best_state, "weighted_score": final_best_score,
        "classes": CLASSES, "height": HEIGHT, "width": WIDTH, "run_name": run_name,
    }, checkpoint_path)
    if WANDB_ENABLED:
        wandb.finish()
    return pd.DataFrame(history), final_best_score


## The scratch model — recover the saved artifact when available

Cell 11 searches all attached Kaggle inputs, its temporary extraction cache, and the current `/kaggle/working` output folder for `model_scratch.pt` or `model_a_scratch.pt`. If found, it copies the artifact into this run's writable checkpoint output folder. If no compatible artifact is found, the fifth model is trained once on a CUDA runtime and saved under `/kaggle/working`.

A recovered `model_a_scratch.pt` may contain an auxiliary `classification_head` that is not used by the current segmentation-only U-Net. Cell 35 ignores only those extra classifier parameters and still requires every segmentation parameter name and shape to match exactly.


In [25]:
def build_unet(encoder_name, encoder_weights, num_classes):
    return smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights, in_channels=1, classes=num_classes, activation=None)


def unet_forward(model, images):
    return model(images)

# Define the same four-way flip TTA before scratch-model validation/selection.
FLIP_VARIANTS = [None, (-1,), (-2,), (-2, -1)]

def apply_flip(tensor, dims):
    return tensor if dims is None else torch.flip(tensor, dims=list(dims))

def make_tta_forward(base_forward_fn):
    def tta_forward(model, images):
        probability_sum = None
        for dims in FLIP_VARIANTS:
            flipped = apply_flip(images, dims)
            logits = base_forward_fn(model, flipped)
            probabilities = torch.sigmoid(logits.float())
            probabilities = apply_flip(probabilities, dims)
            probability_sum = probabilities if probability_sum is None else probability_sum + probabilities
        mean_probabilities = probability_sum / len(FLIP_VARIANTS)
        return torch.logit(mean_probabilities.clamp(1e-6, 1.0 - 1e-6))
    return tta_forward


In [26]:
_dummy_model = build_unet("resnet34", None, N_CLASSES)
_dummy_model.eval()
_dummy_input = torch.randn(1, 1, HEIGHT, WIDTH)
_seg_out = unet_forward(_dummy_model, _dummy_input)
assert _seg_out.shape == (1, N_CLASSES, HEIGHT, WIDTH), _seg_out.shape
print("Scratch UNet smoke test OK —", _seg_out.shape)
del _dummy_model, _dummy_input, _seg_out

Scratch UNet smoke test OK — torch.Size([1, 4, 128, 800])


In [27]:
# Load a saved scratch checkpoint, allowing only an unused auxiliary classification head.
# Cell 11 may have recovered this artifact from an attached Kaggle dataset and copied it to /kaggle/working.
SCRATCH_CHECKPOINT_PATH = CHECKPOINT_DIR / "model_scratch.pt"

def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")


def load_scratch_checkpoint_compatible(model, checkpoint, checkpoint_path):
    """Load all segmentation weights strictly; ignore only extra classifier-head keys."""
    if "model_state_dict" not in checkpoint:
        raise KeyError(f"Scratch checkpoint has no model_state_dict: {checkpoint_path}")

    saved_state = checkpoint["model_state_dict"]
    model_state = model.state_dict()
    saved_keys = set(saved_state)
    model_keys = set(model_state)

    missing = sorted(model_keys - saved_keys)
    extra = sorted(saved_keys - model_keys)
    unexpected_non_classifier = [key for key in extra if not key.startswith("classification_head.")]
    shape_mismatches = [
        (key, tuple(saved_state[key].shape), tuple(model_state[key].shape))
        for key in sorted(model_keys & saved_keys)
        if tuple(saved_state[key].shape) != tuple(model_state[key].shape)
    ]

    if missing or unexpected_non_classifier or shape_mismatches:
        raise RuntimeError(
            "Scratch checkpoint is not compatible with the current segmentation U-Net. "
            f"Missing segmentation keys={missing[:10]}; "
            f"unexpected non-classifier keys={unexpected_non_classifier[:10]}; "
            f"shape mismatches={shape_mismatches[:10]}"
        )

    # Remove only checkpoint-only auxiliary classification parameters.
    segmentation_state = {key: saved_state[key] for key in model_keys}
    result = model.load_state_dict(segmentation_state, strict=True)
    if result.missing_keys or result.unexpected_keys:
        raise RuntimeError(f"Strict scratch segmentation load failed: {result}")

    print("Scratch segmentation checkpoint loaded successfully:", checkpoint_path)
    print("Ignored unused auxiliary classifier keys:", extra if extra else "none")
    print("Matched segmentation parameters:", len(segmentation_state))
    return model


model_scratch = build_unet("resnet34", None, N_CLASSES).to(DEVICE)

if SCRATCH_CHECKPOINT_PATH.exists():
    checkpoint_scratch = torch_load_checkpoint(SCRATCH_CHECKPOINT_PATH)
    saved_classes = checkpoint_scratch.get("classes")
    if saved_classes is not None and list(saved_classes) != CLASSES:
        raise ValueError(f"Scratch checkpoint class order mismatch: {saved_classes} != {CLASSES}")
    saved_h, saved_w = checkpoint_scratch.get("height"), checkpoint_scratch.get("width")
    if saved_h is not None and int(saved_h) != HEIGHT:
        raise ValueError(f"Scratch checkpoint height mismatch: {saved_h}")
    if saved_w is not None and int(saved_w) != WIDTH:
        raise ValueError(f"Scratch checkpoint width mismatch: {saved_w}")

    model_scratch = load_scratch_checkpoint_compatible(
        model_scratch, checkpoint_scratch, SCRATCH_CHECKPOINT_PATH
    )
    model_scratch.eval()
    print("Existing scratch checkpoint reused — skipping training.")
else:
    if DEVICE.type != "cuda":
        raise RuntimeError(
            "No scratch checkpoint was found in the downloaded Kaggle checkpoint dataset, "
            "temporary cache, or current Drive. The fifth scratch model requires a CUDA runtime "
            "for practical training. Switch Colab to a T4 GPU before running this cell."
        )
    print("No scratch checkpoint found — training fresh for 20 epochs.")
    EPOCHS_SCRATCH = 20
    BATCH_SIZE_SCRATCH = 8
    train_loader_scratch = build_train_loader(batch_size=BATCH_SIZE_SCRATCH)
    optimizer_scratch = torch.optim.AdamW(model_scratch.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler_scratch = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_scratch, T_max=len(train_loader_scratch) * EPOCHS_SCRATCH
    )
    scratch_validation_forward = make_tta_forward(unet_forward) if USE_TTA_FOR_CALIBRATION else unet_forward
    history_scratch, score_scratch = train_model(
        model_scratch, unet_forward, train_loader_scratch, cal_loader,
        optimizer_scratch, scheduler_scratch, EPOCHS_SCRATCH, DEVICE,
        SCRATCH_CHECKPOINT_PATH, run_name="v9-scratch-resnet34-unet",
        run_config={"encoder": "resnet34", "pretrained": False, "epochs": EPOCHS_SCRATCH, "role": "built_from_scratch"},
        validation_forward_fn=scratch_validation_forward,
    )
    checkpoint_scratch = torch_load_checkpoint(SCRATCH_CHECKPOINT_PATH)
    model_scratch = load_scratch_checkpoint_compatible(
        model_scratch, checkpoint_scratch, SCRATCH_CHECKPOINT_PATH
    )
    model_scratch.eval()
    print("Scratch model best calibration score:", score_scratch)
    print("Scratch checkpoint saved:", SCRATCH_CHECKPOINT_PATH)


Scratch segmentation checkpoint loaded successfully: /kaggle/working/surfaceguard_v9_2_5_outputs/checkpoints/model_scratch.pt
Ignored unused auxiliary classifier keys: ['classification_head.3.bias', 'classification_head.3.weight']
Matched segmentation parameters: 278
Existing scratch checkpoint reused — skipping training.


## Loading the four existing checkpoints

No retraining — these already cost real GPU hours and already work. Paths follow the layout from your own ensemble notebook's `v4-v5-v6-mit` dataset; adjust `CHECKPOINT_DATASET_DIR` above if yours differs.

In [28]:
def find_checkpoint(filename):
    if filename not in CHECKPOINT_PATHS:
        raise KeyError(f"Unexpected checkpoint filename requested: {filename}")
    path = Path(CHECKPOINT_PATHS[filename])
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint path disappeared: {path}")
    return path


def verify_checkpoint_metadata(checkpoint, label):
    saved_classes = checkpoint.get("classes")
    if saved_classes is not None and list(saved_classes) != CLASSES:
        raise ValueError(f"{label} class order mismatch: {list(saved_classes)} != {CLASSES}")
    saved_h, saved_w = checkpoint.get("height"), checkpoint.get("width")
    if saved_h is not None and int(saved_h) != HEIGHT:
        raise ValueError(f"{label} height mismatch: {saved_h}")
    if saved_w is not None and int(saved_w) != WIDTH:
        raise ValueError(f"{label} width mismatch: {saved_w}")
    if "model_state_dict" not in checkpoint:
        raise KeyError(f"{label} checkpoint has no model_state_dict")


def strict_load_segformer(model, checkpoint, label):
    """
    Load the checkpoint without renaming keys.

    The current model's state_dict expects the same `segformer.stages...`
    namespace present in the checkpoint. The previous conversion routine
    renamed those keys to `segformer.encoder...`, causing the 192/192
    missing/unexpected-key failure.
    """
    verify_checkpoint_metadata(checkpoint, label)
    source_state = checkpoint["model_state_dict"]
    target_state = model.state_dict()

    source_keys = set(source_state)
    target_keys = set(target_state)
    missing = sorted(target_keys - source_keys)
    unexpected = sorted(source_keys - target_keys)
    shape_mismatches = [
        (key, tuple(source_state[key].shape), tuple(target_state[key].shape))
        for key in sorted(source_keys & target_keys)
        if tuple(source_state[key].shape) != tuple(target_state[key].shape)
    ]

    if missing or unexpected or shape_mismatches:
        raise RuntimeError(
            f"{label} direct checkpoint load failed: "
            f"missing={len(missing)}, unexpected={len(unexpected)}, "
            f"shape_mismatches={len(shape_mismatches)}. "
            f"First missing={missing[:8]}, first unexpected={unexpected[:8]}, "
            f"first shape mismatch={shape_mismatches[:5]}. "
            "No key conversion was applied; inspect the checkpoint/model "
            "architecture and Transformers version before changing mappings."
        )

    result = model.load_state_dict(source_state, strict=True)
    if result.missing_keys or result.unexpected_keys:
        raise RuntimeError(f"{label} strict load was not clean: {result}")

    return {
        "epoch": checkpoint.get("epoch"),
        "mean_dice": checkpoint.get("mean_dice"),
        "parameter_count": len(source_state),
        "key_conversion": "none — direct strict load",
    }


# V4 — U-Net / ResNet34.
model_v4 = build_unet("resnet34", None, N_CLASSES).to(DEVICE)
checkpoint_v4 = torch_load_checkpoint(find_checkpoint("best_v4.pt"))
verify_checkpoint_metadata(checkpoint_v4, "V4 U-Net/ResNet34")
model_v4.load_state_dict(checkpoint_v4["model_state_dict"], strict=True)
META_V4 = {"epoch": checkpoint_v4.get("epoch")}
del checkpoint_v4
model_v4.eval()
print("V4 U-Net/ResNet34 loaded strictly")

# V5 — SegFormer-B0. Load checkpoint keys directly; do not rename them.
V5_MODEL_ID = "nvidia/segformer-b0-finetuned-ade-512-512"
config_v5 = SegformerConfig.from_pretrained(V5_MODEL_ID)
config_v5.num_labels = N_CLASSES
config_v5.id2label = {i: name for i, name in enumerate(CLASSES)}
config_v5.label2id = {name: i for i, name in enumerate(CLASSES)}
model_v5 = SegformerForSemanticSegmentation(config_v5)
checkpoint_v5 = torch_load_checkpoint(find_checkpoint("best_v5.pt"))
META_V5 = strict_load_segformer(model_v5, checkpoint_v5, "V5 SegFormer-B0")
del checkpoint_v5
model_v5 = model_v5.to(DEVICE).eval()
print("V5 SegFormer-B0 loaded strictly:", META_V5)

# V6 — DeepLabV3+ / ResNet50.
model_v6 = smp.DeepLabV3Plus(
    encoder_name="resnet50", encoder_weights=None, in_channels=1,
    classes=N_CLASSES, activation=None
).to(DEVICE)
checkpoint_v6 = torch_load_checkpoint(find_checkpoint("best_v6_deeplabv3plus.pt"))
verify_checkpoint_metadata(checkpoint_v6, "V6 DeepLabV3+/ResNet50")
model_v6.load_state_dict(checkpoint_v6["model_state_dict"], strict=True)
META_V6 = {"epoch": checkpoint_v6.get("epoch")}
del checkpoint_v6
model_v6.eval()
print("V6 DeepLabV3+/ResNet50 loaded strictly")

# MiT-B4 / SegFormer-B4. Use the same direct strict-load routine.
MIT_MODEL_ID = "nvidia/segformer-b4-finetuned-ade-512-512"
config_mit = SegformerConfig.from_pretrained(MIT_MODEL_ID)
config_mit.num_labels = N_CLASSES
config_mit.id2label = {i: name for i, name in enumerate(CLASSES)}
config_mit.label2id = {name: i for i, name in enumerate(CLASSES)}
model_mit = SegformerForSemanticSegmentation(config_mit)
checkpoint_mit = torch_load_checkpoint(find_checkpoint("best_mit_b4.pt"))
META_MIT = strict_load_segformer(model_mit, checkpoint_mit, "MiT-B4")
del checkpoint_mit
model_mit = model_mit.to(DEVICE).eval()
print("MiT-B4 loaded strictly:", META_MIT)

print("\nALL FOUR PRETRAINED CHECKPOINTS LOADED AND VERIFIED")


V4 U-Net/ResNet34 loaded strictly


config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

V5 SegFormer-B0 loaded strictly: {'epoch': 7, 'mean_dice': 0.5034020691225941, 'parameter_count': 208, 'key_conversion': 'none — direct strict load'}


V6 DeepLabV3+/ResNet50 loaded strictly


config.json:   0%|          | 0.00/6.89k [00:00<?, ?B/s]

MiT-B4 loaded strictly: {'epoch': 6, 'mean_dice': None, 'parameter_count': 930, 'key_conversion': 'none — direct strict load'}

ALL FOUR PRETRAINED CHECKPOINTS LOADED AND VERIFIED


In [29]:
def resize_segformer_logits(logits):
    if logits.shape[-2:] != (HEIGHT, WIDTH):
        logits = F.interpolate(logits.float(), size=(HEIGHT, WIDTH), mode="bilinear", align_corners=False)
    return logits.float()


def v5_forward(model, images):
    rgb_images = images.repeat(1, 3, 1, 1)
    return resize_segformer_logits(model(pixel_values=rgb_images).logits)


def v6_forward(model, images):
    return model(images)


def mit_forward(model, images):
    rgb_images = images.repeat(1, 3, 1, 1)
    return resize_segformer_logits(model(pixel_values=rgb_images).logits)

## Four-way flip TTA used consistently

On a CUDA runtime, all five models use the same four transforms (identity, horizontal flip, vertical flip, both flips). The same TTA wrapper is used when caching calibration probabilities, evaluating the reserved holdout, and predicting the test set. This avoids calibrating weights/thresholds on single-pass probabilities and then applying them to a differently distributed TTA blend. CPU fallback uses one pass consistently. This is more inference work than the earlier final-only TTA variant.
Each model averages probabilities across the four inverse-aligned views (rather than averaging logits), matching the probability-averaging convention used in the V14 inference pipeline. The mean probability is converted back to logits only to preserve V9.1 downstream interfaces.


In [30]:
FLIP_VARIANTS = [None, (-1,), (-2,), (-2, -1)]


def apply_flip(tensor, dims):
    return tensor if dims is None else torch.flip(tensor, dims=list(dims))


def make_tta_forward(base_forward_fn):
    def tta_forward(model, images):
        probability_sum = None
        for dims in FLIP_VARIANTS:
            flipped = apply_flip(images, dims)
            logits = base_forward_fn(model, flipped)
            probabilities = torch.sigmoid(logits.float())
            probabilities = apply_flip(probabilities, dims)
            probability_sum = probabilities if probability_sum is None else probability_sum + probabilities
        mean_probabilities = probability_sum / len(FLIP_VARIANTS)
        # The rest of V9.1 expects logits and applies sigmoid; logit() preserves
        # the probability-averaged TTA result without changing the downstream API.
        return torch.logit(mean_probabilities.clamp(1e-6, 1.0 - 1e-6))
    return tta_forward


BASE_FORWARD_FNS = {"v4": unet_forward, "v5": v5_forward, "v6": v6_forward, "mit": mit_forward, "scratch": unet_forward}
TTA_FORWARD_FNS = {name: make_tta_forward(fn) for name, fn in BASE_FORWARD_FNS.items()}

In [31]:
_dummy_input = torch.randn(1, 1, HEIGHT, WIDTH, device=DEVICE)
_models_for_check = {"v4": model_v4, "v5": model_v5, "v6": model_v6, "mit": model_mit, "scratch": model_scratch}
with torch.inference_mode():
    for name, model in _models_for_check.items():
        base_out = BASE_FORWARD_FNS[name](model, _dummy_input)
        tta_out = TTA_FORWARD_FNS[name](model, _dummy_input)
        assert base_out.shape == (1, N_CLASSES, HEIGHT, WIDTH), (name, "base", base_out.shape)
        assert tta_out.shape == (1, N_CLASSES, HEIGHT, WIDTH), (name, "tta", tta_out.shape)
print("All five models' base and TTA forward passes OK")
del _dummy_input, _models_for_check

All five models' base and TTA forward passes OK


## Caching probabilities for calibration and holdout

Each model's probabilities are computed once per split and memory-mapped to local Kaggle temporary disk. CUDA runs use the same four-way flip TTA as final test inference; CPU fallback uses a single pass. The blend search reads these cached arrays rather than rerunning the networks for every candidate weight vector.

In [32]:
def cache_masks(dataframe, loader, split_name):
    n_items = len(dataframe)
    path = CACHE_DIR / f"{split_name}_masks.npy"
    memmap = np.lib.format.open_memmap(path, mode="w+", dtype=np.uint8, shape=(n_items, N_CLASSES, HEIGHT, WIDTH))
    cursor = 0
    for _, masks, _ in tqdm(loader, desc=f"Caching {split_name} masks"):
        batch_size = masks.shape[0]
        memmap[cursor:cursor + batch_size] = masks.numpy().astype(np.uint8)
        cursor += batch_size
    memmap.flush()
    if cursor != n_items:
        raise RuntimeError(f"{split_name} masks: cached {cursor}, expected {n_items}")
    return np.load(path, mmap_mode="r")


def cache_probabilities(name, forward_fn, model, dataframe, loader, split_name):
    n_items = len(dataframe)
    path = CACHE_DIR / f"{split_name}_{name}_probs.npy"
    memmap = np.lib.format.open_memmap(path, mode="w+", dtype=np.float16, shape=(n_items, N_CLASSES, HEIGHT, WIDTH))
    cursor = 0
    model.eval()
    with torch.inference_mode():
        for images, _, _ in tqdm(loader, desc=f"Caching {split_name}/{name}"):
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                logits = forward_fn(model, images)
            probs = torch.sigmoid(logits.float()).cpu().numpy().astype(np.float16)
            batch_size = probs.shape[0]
            memmap[cursor:cursor + batch_size] = probs
            cursor += batch_size
    memmap.flush()
    if cursor != n_items:
        raise RuntimeError(f"{split_name}/{name}: cached {cursor}, expected {n_items}")
    return np.load(path, mmap_mode="r")

In [33]:
cal_masks = cache_masks(cal_df, cal_loader, "cal")
holdout_masks = cache_masks(holdout_df, holdout_loader, "holdout")

MODELS_BY_NAME = {"v4": model_v4, "v5": model_v5, "v6": model_v6, "mit": model_mit, "scratch": model_scratch}

# Match calibration and holdout predictions to the exact inference mode used on test images.
CALIBRATION_FORWARD_FNS = TTA_FORWARD_FNS if USE_TTA_FOR_CALIBRATION else BASE_FORWARD_FNS
MODEL_FORWARD_FNS = {name: (model, CALIBRATION_FORWARD_FNS[name]) for name, model in MODELS_BY_NAME.items()}

cal_probs = {}
holdout_probs = {}
for name, (model, forward_fn) in MODEL_FORWARD_FNS.items():
    cal_probs[name] = cache_probabilities(name, forward_fn, model, cal_df, cal_loader, "cal")
    holdout_probs[name] = cache_probabilities(name, forward_fn, model, holdout_df, holdout_loader, "holdout")

print("All five models cached for calibration and holdout using the final inference mode.")

Caching cal masks:   0%|          | 0/1367 [00:00<?, ?it/s]

Caching holdout masks:   0%|          | 0/912 [00:00<?, ?it/s]

Caching cal/v4:   0%|          | 0/1367 [00:00<?, ?it/s]

Caching holdout/v4:   0%|          | 0/912 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e7980506f20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e7980506f20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Caching cal/v5:   0%|          | 0/1367 [00:00<?, ?it/s]

... (7 outputs left out)

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e7980506f20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e7980506f20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Caching holdout/scratch:   0%|          | 0/912 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e7980506f20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e7980506f20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

All five models cached for calibration and holdout using the final inference mode.


## Blend evaluation — class-aware from the start

Generalized to let each class use its own weight vector across models, not one vector shared by all four — this is the technique your own `0_63881` notebook proved out (holdout 0.707 → 0.717). `evaluate_blend` below is the special case where every class happens to get the same vector; `evaluate_class_aware` is the real implementation both call into, so there's exactly one place this math can be wrong, and it's the one already checked against an independent brute-force reference.

For each image/class pair, every threshold's predicted-pixel-count and true-positive count is read off a single sort of that pair's probability map via `searchsorted`, instead of rebuilding a full-resolution boolean mask once per threshold candidate — this is what makes searching many blend-weight candidates across a 91-value threshold grid finish in minutes rather than hours.

In [34]:
def evaluate_class_aware(weight_matrix, probs_dict, masks_array, model_order, thresholds, classes, height, width):
    """Histogram-based per-class threshold search for the pooled official metric.

    Bins are computed from cached probabilities in small chunks. This avoids sorting
    every 128x800 probability map for every blend candidate, which would make the
    five-model grid search unnecessarily slow.
    """
    num_c = len(classes)
    pixels_per_pair = height * width
    n_items = masks_array.shape[0]
    n_thresholds = len(thresholds)
    n_bins = n_thresholds + 1
    chunk_size = 8
    pos_dice_sum = np.zeros((n_thresholds, num_c), dtype=np.float64)
    fp_pixels_sum = np.zeros((n_thresholds, num_c), dtype=np.float64)
    empty_correct_sum = np.zeros((n_thresholds, num_c), dtype=np.float64)
    pos_counts = np.zeros(num_c, dtype=np.int64)
    empty_counts = np.zeros(num_c, dtype=np.int64)

    class_weight_vectors = [np.asarray(weight_matrix[classes[c]], dtype=np.float64) for c in range(num_c)]
    for c, weights in enumerate(class_weight_vectors):
        if (len(weights) != len(model_order) or not np.isfinite(weights).all()
                or np.any(weights < -1e-9) or not np.isclose(weights.sum(), 1.0, atol=1e-6)):
            raise ValueError(f"Invalid blend weights for {classes[c]}: {weights}")

    for c in range(num_c):
        weights = class_weight_vectors[c]
        active_models = [(float(w), name) for w, name in zip(weights, model_order) if w > 0]
        for chunk_start in range(0, n_items, chunk_size):
            row_indices = np.arange(chunk_start, min(chunk_start + chunk_size, n_items))
            batch_n = len(row_indices)
            combined = np.zeros((batch_n, height, width), dtype=np.float32)
            for weight, name in active_models:
                combined += weight * np.asarray(probs_dict[name][row_indices, c], dtype=np.float32)

            targets = np.asarray(masks_array[row_indices, c] > 0, dtype=bool)
            target_counts = np.count_nonzero(targets, axis=(1, 2))
            positive = target_counts > 0
            empty = ~positive
            pos_counts[c] += int(positive.sum())
            empty_counts[c] += int(empty.sum())

            combined_flat = combined.reshape(batch_n, -1)
            targets_flat = targets.reshape(batch_n, -1)
            bins = np.searchsorted(thresholds, combined_flat, side="right").astype(np.uint8)
            hist_all = np.zeros((batch_n, n_bins), dtype=np.uint32)
            hist_positive = np.zeros_like(hist_all)
            for local_row in range(batch_n):
                hist_all[local_row] = np.bincount(bins[local_row], minlength=n_bins).astype(np.uint32)
                hist_positive[local_row] = np.bincount(
                    bins[local_row][targets_flat[local_row]], minlength=n_bins
                ).astype(np.uint32)

            suffix_all = np.cumsum(hist_all[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
            suffix_positive = np.cumsum(hist_positive[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
            for threshold_idx in range(n_thresholds):
                predicted_counts = suffix_all[:, threshold_idx + 1].astype(np.float64)
                intersection_counts = suffix_positive[:, threshold_idx + 1].astype(np.float64)
                if positive.any():
                    pair_dice = 2.0 * intersection_counts[positive] / np.maximum(
                        predicted_counts[positive] + target_counts[positive], 1.0
                    )
                    pos_dice_sum[threshold_idx, c] += float(pair_dice.sum())
                if empty.any():
                    fp_pixels_sum[threshold_idx, c] += float(predicted_counts[empty].sum())
                    empty_correct_sum[threshold_idx, c] += float((predicted_counts[empty] == 0).sum())

    total_positive = int(pos_counts.sum())
    total_empty = int(empty_counts.sum())
    if total_positive == 0 or total_empty == 0:
        raise ValueError(f"Metric needs positive and empty image-class pairs; got positive={total_positive}, empty={total_empty}.")
    missing_positive = [classes[c] for c in range(num_c) if pos_counts[c] == 0]
    missing_empty = [classes[c] for c in range(num_c) if empty_counts[c] == 0]
    if missing_positive or missing_empty:
        raise ValueError(f"Per-class threshold search needs both pair types. Missing positive={missing_positive}; missing empty={missing_empty}.")

    contributions = np.zeros((n_thresholds, num_c), dtype=np.float64)
    for c in range(num_c):
        contributions[:, c] = (
            0.70 * pos_dice_sum[:, c] / total_positive
            + 0.20 * (empty_counts[c] * pixels_per_pair - fp_pixels_sum[:, c]) / (total_empty * pixels_per_pair)
            + 0.10 * empty_correct_sum[:, c] / total_empty
        )

    selected = contributions.argmax(axis=0)
    per_class = []
    for c, name in enumerate(classes):
        idx = int(selected[c])
        dice = float(pos_dice_sum[idx, c] / pos_counts[c])
        fp_score = float(1.0 - fp_pixels_sum[idx, c] / (empty_counts[c] * pixels_per_pair))
        empty_acc = float(empty_correct_sum[idx, c] / empty_counts[c])
        per_class.append({
            "class": name, "threshold": float(thresholds[idx]),
            "positive_pair_dice": dice, "false_positive_score": fp_score,
            "empty_accuracy": empty_acc,
            "class_local_score": float(0.70 * dice + 0.20 * fp_score + 0.10 * empty_acc),
            "weighted_score": float(contributions[idx, c]),
            "grid_edge": bool(idx == 0 or idx == n_thresholds - 1),
            "positive_pairs": int(pos_counts[c]), "empty_pairs": int(empty_counts[c]),
        })
    official_score = float(sum(row["weighted_score"] for row in per_class))
    thresholds_dict = {row["class"]: row["threshold"] for row in per_class}
    return official_score, thresholds_dict, per_class


def evaluate_blend(weights, probs_dict, masks_array, model_order, thresholds, classes, height, width):
    weight_matrix = {class_name: weights for class_name in classes}
    return evaluate_class_aware(weight_matrix, probs_dict, masks_array, model_order, thresholds, classes, height, width)


In [35]:
# Deliberately asymmetric class support: defect_a/defect_d have 1 positive each; defect_b/defect_c have 2 each.
# The class-specific performance also differs, so this test catches accidental
# replacement of the pooled competition metric by a macro-average of class scores.
_toy_masks = np.zeros((4, N_CLASSES, 4, 4), dtype=np.uint8)
_toy_masks[0, :, :2, :2] = 1
_toy_masks[1, 1, :2, :2] = 1
_toy_masks[2, 2, :2, :2] = 1
_toy_a = np.full((4, N_CLASSES, 4, 4), 0.1, dtype=np.float16)
_toy_b = np.zeros_like(_toy_a)
# defect_a / defect_d predict the entire canvas (poor), defect_b is exact,
# defect_c predicts empty masks (strong on negatives but misses positives).
_toy_a[:, 0, :, :] = 0.9
_toy_a[:, 3, :, :] = 0.9
_toy_a[:, 1, :, :] = 0.1
_toy_a[:, 1, :, :][ _toy_masks[:, 1] > 0 ] = 0.9
_toy_a[:, 2, :, :] = 0.1
_toy_probs = {"a": _toy_a, "b": _toy_b}
_toy_weight_matrix = {name: [1.0, 0.0] for name in CLASSES}
_toy_score, _toy_thresholds, _toy_per_class = evaluate_class_aware(
    _toy_weight_matrix, _toy_probs, _toy_masks, ["a", "b"],
    np.array([0.5], dtype=np.float32), CLASSES, 4, 4
)
# Independent implementation of the competition's pooled formula.
_positive_dice = []
_empty_fp_fraction = []
_empty_correct = []
for image_idx in range(len(_toy_masks)):
    for class_idx in range(N_CLASSES):
        pred = _toy_a[image_idx, class_idx] >= 0.5
        target = _toy_masks[image_idx, class_idx] > 0
        if target.any():
            _positive_dice.append(2.0 * np.logical_and(pred, target).sum() / max(pred.sum() + target.sum(), 1))
        else:
            _empty_fp_fraction.append(pred.sum() / float(4 * 4))
            _empty_correct.append(float(pred.sum() == 0))
_expected_official = (
    0.70 * np.mean(_positive_dice)
    + 0.20 * (1.0 - np.mean(_empty_fp_fraction))
    + 0.10 * np.mean(_empty_correct)
)
_macro_class_score = float(np.mean([row["class_local_score"] for row in _toy_per_class]))
assert np.isclose(_toy_score, _expected_official, atol=1e-9), (_toy_score, _expected_official)
assert not np.isclose(_toy_score, _macro_class_score, atol=1e-4), (_toy_score, _macro_class_score)
assert np.isclose(_toy_score, sum(row["weighted_score"] for row in _toy_per_class), atol=1e-9)
assert set(_toy_thresholds) == set(CLASSES)
print("Pooled official-metric asymmetric-support test: PASSED")
print("Pooled score:", round(_toy_score, 6), "| class-macro score (intentionally different):", round(_macro_class_score, 6))
del _toy_masks, _toy_a, _toy_b, _toy_probs, _toy_weight_matrix, _toy_score, _toy_thresholds, _toy_per_class


Pooled official-metric asymmetric-support test: PASSED
Pooled score: 0.446667 | class-macro score (intentionally different): 0.465


## Blend weight search — joint over all five models, then made class-aware

Coarse grid (0.25 steps, 70 combinations across the five-model simplex) then local refinement (0.125 steps around the best) — re-derived on this notebook's own calibration split rather than assuming weights from a different run transfer unchanged. Every candidate's full per-class breakdown is kept, not just its overall score, because that's what the class-aware extraction step right after this needs.

In [36]:
FULL_MODEL_ORDER = ["v4", "v5", "v6", "mit", "scratch"]
LEVELS = [0.0, 0.25, 0.5, 0.75, 1.0]

search_rows = []
for w_v5 in LEVELS:
    for w_v6 in LEVELS:
        for w_mit in LEVELS:
            for w_scratch in LEVELS:
                if round(w_v5 + w_v6 + w_mit + w_scratch, 3) > 1.0:
                    continue
                w_v4 = round(1.0 - w_v5 - w_v6 - w_mit - w_scratch, 3)
                weights = [w_v4, w_v5, w_v6, w_mit, w_scratch]
                score, thresholds, per_class = evaluate_blend(
                    weights, cal_probs, cal_masks, FULL_MODEL_ORDER, THRESHOLD_GRID, CLASSES, HEIGHT, WIDTH
                )
                search_rows.append({"stage": "coarse", "weights": weights, "score": score, "thresholds": thresholds, "per_class": per_class})

print(f"Coarse search: {len(search_rows)} candidates evaluated")
best_coarse = max(search_rows, key=lambda r: r["score"])
print("Best coarse weights:", dict(zip(FULL_MODEL_ORDER, best_coarse["weights"])), "| score:", round(best_coarse["score"], 5))

Coarse search: 70 candidates evaluated
Best coarse weights: {'v4': 0.0, 'v5': 0.25, 'v6': 0.25, 'mit': 0.5, 'scratch': 0.0} | score: 0.71816


In [37]:
refine_step = 0.125
refine_candidates = set()
base_w = best_coarse["weights"]
for d5 in (-refine_step, 0.0, refine_step):
    for d6 in (-refine_step, 0.0, refine_step):
        for dm in (-refine_step, 0.0, refine_step):
            for ds in (-refine_step, 0.0, refine_step):
                w_v5 = min(max(base_w[1] + d5, 0.0), 1.0)
                w_v6 = min(max(base_w[2] + d6, 0.0), 1.0)
                w_mit = min(max(base_w[3] + dm, 0.0), 1.0)
                w_scratch = min(max(base_w[4] + ds, 0.0), 1.0)
                if w_v5 + w_v6 + w_mit + w_scratch > 1.0:
                    continue
                w_v4 = round(1.0 - w_v5 - w_v6 - w_mit - w_scratch, 4)
                refine_candidates.add((round(w_v4, 4), round(w_v5, 4), round(w_v6, 4), round(w_mit, 4), round(w_scratch, 4)))

for weights in refine_candidates:
    score, thresholds, per_class = evaluate_blend(
        list(weights), cal_probs, cal_masks, FULL_MODEL_ORDER, THRESHOLD_GRID, CLASSES, HEIGHT, WIDTH
    )
    search_rows.append({"stage": "refine", "weights": list(weights), "score": score, "thresholds": thresholds, "per_class": per_class})

best_global = max(search_rows, key=lambda r: r["score"])
print(f"Refine search: {len(refine_candidates)} additional candidates | total evaluated: {len(search_rows)}")
print("Best global weights:", dict(zip(FULL_MODEL_ORDER, best_global["weights"])), "| score:", round(best_global["score"], 5))

Refine search: 27 additional candidates | total evaluated: 97
Best global weights: {'v4': 0.125, 'v5': 0.125, 'v6': 0.25, 'mit': 0.5, 'scratch': 0.0} | score: 0.72185


## Class-aware extraction

For each class independently: scan every candidate already evaluated above, and adopt whichever one maximizes that class's own contribution — but only if it's within a conservative distance of the global blend (`MAX_CLASS_L1_DISTANCE`) and clears a minimum required gain (set higher for `defect_b`, which has the fewest positive examples in validation and is the easiest class to overfit calibration noise on). Otherwise that class falls back to the global weights. This is the technique from your own `0_63881` notebook, generalized to five models.

In [38]:
GLOBAL_WEIGHTS = np.asarray(best_global["weights"], dtype=np.float64)
MAX_CLASS_L1_DISTANCE = 0.50
MIN_CLASS_GAIN = {"defect_a": 0.00025, "defect_b": 0.00075, "defect_c": 0.00025, "defect_d": 0.00025}

# V14's already-tested four-model class-aware blend, extended with zero scratch weight.
# This is an anchor candidate, not a promise that its old public score will repeat.
V14_ANCHOR_MATRIX = {
    "defect_a": [0.25, 0.25, 0.00, 0.50, 0.00],
    "defect_b": [0.25, 0.25, 0.25, 0.25, 0.00],
    "defect_c": [0.00, 0.125, 0.125, 0.75, 0.00],
    "defect_d": [0.125, 0.125, 0.00, 0.75, 0.00],
}
anchor_score, anchor_thresholds, anchor_per_class = evaluate_class_aware(
    V14_ANCHOR_MATRIX, cal_probs, cal_masks, FULL_MODEL_ORDER,
    THRESHOLD_GRID, CLASSES, HEIGHT, WIDTH
)
anchor_per_class = {row["class"]: row for row in anchor_per_class}

CLASS_WEIGHT_MATRIX = {}
FINAL_THRESHOLDS = {}
class_selection_log = {}
global_per_class = {row["class"]: row for row in best_global["per_class"]}

for class_name in CLASSES:
    baseline_score = global_per_class[class_name]["weighted_score"]
    eligible = [
        row for row in search_rows
        if float(np.abs(np.asarray(row["weights"]) - GLOBAL_WEIGHTS).sum()) <= MAX_CLASS_L1_DISTANCE + 1e-9
    ]
    best_row_for_class = max(
        eligible,
        key=lambda row: next(p["weighted_score"] for p in row["per_class"] if p["class"] == class_name)
    )
    best_class_score = next(p["weighted_score"] for p in best_row_for_class["per_class"] if p["class"] == class_name)
    best_weights = list(best_row_for_class["weights"])
    best_threshold = next(p["threshold"] for p in best_row_for_class["per_class"] if p["class"] == class_name)
    source = "near-global class-specific candidate"

    # Consider the known V14 row even if it falls outside the local simplex neighbourhood.
    anchor_row = anchor_per_class[class_name]
    anchor_score_for_class = float(anchor_row["weighted_score"])
    anchor_is_better_than_global = anchor_score_for_class > baseline_score + 1e-12
    if anchor_score_for_class > best_class_score:
        best_class_score = anchor_score_for_class
        best_weights = list(V14_ANCHOR_MATRIX[class_name])
        best_threshold = anchor_row["threshold"]
        source = "V14 four-model anchor"

    gain = best_class_score - baseline_score
    # The previously public-tested V14 configuration is allowed as a fallback candidate
    # whenever it beats the global row on this calibration set, even if the small gain
    # does not clear the extra class-specific-gain threshold.
    if source == "V14 four-model anchor" and anchor_is_better_than_global:
        CLASS_WEIGHT_MATRIX[class_name] = best_weights
        FINAL_THRESHOLDS[class_name] = best_threshold
        class_selection_log[class_name] = {"decision": "V14 anchor retained (better than global on calibration)", "gain": gain}
    elif gain >= MIN_CLASS_GAIN[class_name]:
        CLASS_WEIGHT_MATRIX[class_name] = best_weights
        FINAL_THRESHOLDS[class_name] = best_threshold
        class_selection_log[class_name] = {"decision": source + " accepted", "gain": gain}
    else:
        CLASS_WEIGHT_MATRIX[class_name] = list(GLOBAL_WEIGHTS)
        FINAL_THRESHOLDS[class_name] = global_per_class[class_name]["threshold"]
        class_selection_log[class_name] = {"decision": "retained global weights (gain below minimum)", "gain": gain}

print("CLASS-AWARE WEIGHT SELECTION (POOLED OFFICIAL METRIC)")
for class_name in CLASSES:
    print(f"{class_name}: {dict(zip(FULL_MODEL_ORDER, CLASS_WEIGHT_MATRIX[class_name]))} | {class_selection_log[class_name]['decision']} (gain={class_selection_log[class_name]['gain']:.6f})")

classwise_score, _, classwise_per_class = evaluate_class_aware(
    CLASS_WEIGHT_MATRIX, cal_probs, cal_masks, FULL_MODEL_ORDER,
    THRESHOLD_GRID, CLASSES, HEIGHT, WIDTH
)
print("\nClass-aware pooled calibration score:", round(classwise_score, 6))
print("V14 anchor pooled calibration score:", round(anchor_score, 6))
print("Global-only pooled calibration score:", round(best_global["score"], 6))


CLASS-AWARE WEIGHT SELECTION (POOLED OFFICIAL METRIC)
defect_a: {'v4': np.float64(0.125), 'v5': np.float64(0.125), 'v6': np.float64(0.25), 'mit': np.float64(0.5), 'scratch': np.float64(0.0)} | retained global weights (gain below minimum) (gain=0.000000)
defect_b: {'v4': 0.25, 'v5': 0.125, 'v6': 0.125, 'mit': 0.375, 'scratch': 0.125} | near-global class-specific candidate accepted (gain=0.003397)
defect_c: {'v4': 0.0, 'v5': 0.125, 'v6': 0.25, 'mit': 0.625, 'scratch': 0.0} | near-global class-specific candidate accepted (gain=0.001376)
defect_d: {'v4': 0.125, 'v5': 0.125, 'v6': 0.0, 'mit': 0.75, 'scratch': 0.0} | V14 anchor retained (better than global on calibration) (gain=0.000582)

Class-aware pooled calibration score: 0.727208
V14 anchor pooled calibration score: 0.725482
Global-only pooled calibration score: 0.721853


## Honest holdout evaluation

`HOLDOUT_IDX`-equivalent data that was never touched during the search above. Whatever this reports is the most trustworthy local estimate of what the next submission will score.

In [39]:
def evaluate_fixed_class_aware(weight_matrix, thresholds_dict, probs_dict, masks_array, model_order, classes, height, width):
    """Evaluate fixed calibration-selected settings on holdout using the pooled official metric."""
    pixels_per_pair = height * width
    class_weight_vectors = {c: np.asarray(weight_matrix[c], dtype=np.float64) for c in classes}
    pos_dice_sum = {c: 0.0 for c in classes}
    fp_pixels_sum = {c: 0.0 for c in classes}
    empty_correct_sum = {c: 0 for c in classes}
    pos_counts = {c: 0 for c in classes}
    empty_counts = {c: 0 for c in classes}

    for i in range(masks_array.shape[0]):
        gt_stack = np.asarray(masks_array[i]) >= 1
        for c_idx, class_name in enumerate(classes):
            weights = class_weight_vectors[class_name]
            if len(weights) != len(model_order) or not np.isfinite(weights).all() or np.any(weights < 0) or not np.isclose(weights.sum(), 1.0, atol=1e-6):
                raise ValueError(f"Invalid holdout blend weights for {class_name}: {weights}")
            blended = np.zeros((height, width), dtype=np.float32)
            for weight, model_name in zip(weights, model_order):
                if weight > 0:
                    blended += weight * np.asarray(probs_dict[model_name][i, c_idx], dtype=np.float32)
            predicted = blended >= float(thresholds_dict[class_name])
            target = gt_stack[c_idx]
            target_count = int(target.sum())
            predicted_count = int(predicted.sum())
            if target_count > 0:
                intersection = int(np.logical_and(predicted, target).sum())
                pos_dice_sum[class_name] += 2.0 * intersection / max(predicted_count + target_count, 1)
                pos_counts[class_name] += 1
            else:
                fp_pixels_sum[class_name] += predicted_count
                empty_correct_sum[class_name] += int(predicted_count == 0)
                empty_counts[class_name] += 1

    total_positive = sum(pos_counts.values())
    total_empty = sum(empty_counts.values())
    if total_positive == 0 or total_empty == 0:
        raise ValueError(f"Holdout needs positive and empty image-class pairs; got {total_positive} and {total_empty}")

    per_class = []
    for class_name in classes:
        if pos_counts[class_name] == 0 or empty_counts[class_name] == 0:
            raise ValueError(f"Holdout has no positive or no empty pairs for {class_name}")
        dice = pos_dice_sum[class_name] / pos_counts[class_name]
        fp_score = 1.0 - fp_pixels_sum[class_name] / (empty_counts[class_name] * pixels_per_pair)
        empty_accuracy = empty_correct_sum[class_name] / empty_counts[class_name]
        contribution = (
            0.70 * pos_dice_sum[class_name] / total_positive
            + 0.20 * (empty_counts[class_name] * pixels_per_pair - fp_pixels_sum[class_name]) / (total_empty * pixels_per_pair)
            + 0.10 * empty_correct_sum[class_name] / total_empty
        )
        per_class.append({
            "class": class_name,
            "threshold": float(thresholds_dict[class_name]),
            "positive_pair_dice": float(dice),
            "false_positive_score": float(fp_score),
            "empty_accuracy": float(empty_accuracy),
            "class_local_score": float(0.70 * dice + 0.20 * fp_score + 0.10 * empty_accuracy),
            "weighted_score": float(contribution),
            "positive_pairs": int(pos_counts[class_name]),
            "empty_pairs": int(empty_counts[class_name]),
        })
    official_score = float(sum(row["weighted_score"] for row in per_class))
    return official_score, per_class


holdout_score, holdout_per_class = evaluate_fixed_class_aware(
    CLASS_WEIGHT_MATRIX, FINAL_THRESHOLDS, holdout_probs, holdout_masks,
    FULL_MODEL_ORDER, CLASSES, HEIGHT, WIDTH
)
print("RESERVED HOLDOUT RESULTS — calibration-selected weights and thresholds held fixed")
print("=" * 76)
display(pd.DataFrame(holdout_per_class).set_index("class").round(6))
print("\nHoldout pooled official weighted score:", round(holdout_score, 6))

global_only_matrix = {class_name: list(GLOBAL_WEIGHTS) for class_name in CLASSES}
global_holdout_score, global_holdout_per_class = evaluate_fixed_class_aware(
    global_only_matrix, best_global["thresholds"], holdout_probs, holdout_masks,
    FULL_MODEL_ORDER, CLASSES, HEIGHT, WIDTH
)
print("Global-blend-only pooled holdout score:", round(global_holdout_score, 6))
print("Class-aware gain on holdout:", round(holdout_score - global_holdout_score, 6))


RESERVED HOLDOUT RESULTS — calibration-selected weights and thresholds held fixed


,threshold,positive_pair_dice,false_positive_score,empty_accuracy,class_local_score,weighted_score,positive_pairs,empty_pairs
class,,,,,,,,
defect_a,0.27,0.445977,0.999421,0.772121,0.589280,0.121397,87,825
defect_b,0.43,0.365034,0.999853,0.925508,0.548045,0.095923,26,886
defect_c,0.13,0.638506,0.996633,0.751712,0.721452,0.305431,328,584
defect_d,0.25,0.772466,0.999029,0.948320,0.835364,0.203185,138,774



Holdout pooled official weighted score: 0.725936
Global-blend-only pooled holdout score: 0.720239
Class-aware gain on holdout: 0.005698


## Ensemble inference helper — same class-aware blend and same TTA mode

The selected per-class weights and thresholds are learned on calibration predictions. The reserved holdout and final test predictions use the same model forward path (four-way flip TTA on GPU, single-pass on CPU).

In [40]:
def predict_ensemble_on_loader(weight_matrix, model_order, loader, use_tta):
    forward_lookup = TTA_FORWARD_FNS if use_tta else BASE_FORWARD_FNS
    class_weight_vectors = {c: np.asarray(weight_matrix[c], dtype=np.float64) for c in CLASSES}
    needed_models = [
        name for i, name in enumerate(model_order)
        if any(class_weight_vectors[c][i] > 0 for c in CLASSES)
    ]
    predictions = {}
    with torch.inference_mode():
        for image_ids, images in tqdm(loader, desc=f"Ensemble inference (TTA={use_tta})"):
            images = images.to(DEVICE, non_blocking=True)
            per_model_probs = {}
            for name in needed_models:
                model = MODELS_BY_NAME[name]
                model.eval()
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                    logits = forward_lookup[name](model, images)
                per_model_probs[name] = torch.sigmoid(logits.float())

            batch_size = images.shape[0]
            blended = torch.zeros((batch_size, N_CLASSES, HEIGHT, WIDTH), device=DEVICE)
            for c_idx, class_name in enumerate(CLASSES):
                w_vec = class_weight_vectors[class_name]
                for m_idx, name in enumerate(model_order):
                    w = w_vec[m_idx]
                    if w > 0 and name in per_model_probs:
                        blended[:, c_idx] += w * per_model_probs[name][:, c_idx]

            for i, image_id in enumerate(image_ids):
                predictions[image_id] = blended[i].cpu().numpy().astype(np.float16)
    return predictions

In [41]:
TEST_IDS = sample_submission["image_id"].astype(str).drop_duplicates().tolist()
if len(TEST_IDS) != 3000 or len(sample_submission) != 12000:
    raise ValueError(f"Unexpected test template size: {len(TEST_IDS)} images, {len(sample_submission)} rows")

test_dataset = TestDefectDataset(TEST_IDS, TEST_IMAGE_DIR, HEIGHT, WIDTH)
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=EVAL_NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)
print("Test images:", len(TEST_IDS), "| batch size:", EVAL_BATCH_SIZE, "| inference TTA:", USE_TTA_AT_FINAL_INFERENCE)

Test images: 3000 | batch size: 1 | inference TTA: True


## Final test inference and submission

The submission builder now audits exact image/class prediction coverage before handling empty masks. A missing prediction is a hard error; only a genuinely empty predicted mask is written as a single-space RLE value. The saved CSV is reloaded and checked for nulls, blank identifiers, pair coverage, ordering, duplicates, class counts and valid RLE.


In [42]:
USE_PSEUDO_LABELING = False  # Keep False for the final candidate; enabling uses the reserved holdout for selection.
PSEUDO_LABEL_HIGH = 0.90
PSEUDO_LABEL_LOW = 0.05
PSEUDO_LABEL_MIN_CONFIDENT_FRACTION = 0.98
PSEUDO_LABEL_EPOCHS = 2
PSEUDO_LABEL_LR = 2e-5

In [43]:
def masked_segmentation_loss(logits, targets, confidence_mask):
    probabilities = torch.sigmoid(logits.float())
    targets = targets.float()
    confidence_mask = confidence_mask.float()

    bce_per_pixel = F.binary_cross_entropy_with_logits(logits.float(), targets, reduction="none")
    bce = (bce_per_pixel * confidence_mask).sum() / confidence_mask.sum().clamp_min(1.0)

    dims = (0, 2, 3)
    eps = 1e-6
    tp = (probabilities * targets * confidence_mask).sum(dim=dims)
    fp = (probabilities * (1 - targets) * confidence_mask).sum(dim=dims)
    fn = ((1 - probabilities) * targets * confidence_mask).sum(dim=dims)
    tversky = (tp + eps) / (tp + 0.3 * fp + 0.7 * fn + eps)
    tversky_loss = ((1 - tversky).clamp_min(0) ** 0.75).mean()
    return 0.5 * bce + 0.5 * tversky_loss


class PseudoLabelDataset(Dataset):
    def __init__(self, image_ids, image_dir, height, width, pseudo_labels, confidence_masks):
        self.image_ids = list(image_ids)
        self.image_dir = Path(image_dir)
        self.height = height
        self.width = width
        self.pseudo_labels = pseudo_labels
        self.confidence_masks = confidence_masks

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image = load_and_normalize(self.image_dir / f"{self.image_ids[idx]}.jpg", self.height, self.width)
        image = normalize_inplace(image)
        image_tensor = torch.from_numpy(np.ascontiguousarray(image)).unsqueeze(0).float()
        label_tensor = torch.from_numpy(np.ascontiguousarray(self.pseudo_labels[idx])).float()
        mask_tensor = torch.from_numpy(np.ascontiguousarray(self.confidence_masks[idx])).float()
        return image_tensor, label_tensor, mask_tensor

In [44]:
if USE_PSEUDO_LABELING:
    test_ensemble_probs = predict_ensemble_on_loader(CLASS_WEIGHT_MATRIX, FULL_MODEL_ORDER, test_loader, use_tta=USE_TTA_AT_FINAL_INFERENCE)

    pseudo_ids, pseudo_labels_list, confidence_masks_list = [], [], []
    for image_id in TEST_IDS:
        probs = np.asarray(test_ensemble_probs[image_id], dtype=np.float32)
        confident_positive = probs >= PSEUDO_LABEL_HIGH
        confident_negative = probs <= PSEUDO_LABEL_LOW
        confidence_mask = (confident_positive | confident_negative).astype(np.float32)
        if confidence_mask.mean() >= PSEUDO_LABEL_MIN_CONFIDENT_FRACTION:
            pseudo_ids.append(image_id)
            pseudo_labels_list.append(confident_positive.astype(np.float32))
            confidence_masks_list.append(confidence_mask)

    print(f"Pseudo-labeled images: {len(pseudo_ids)} / {len(TEST_IDS)}")

    if len(pseudo_ids) < 50:
        print("Too few confident pseudo-labeled images to fine-tune on safely — skipping.")
    else:
        pseudo_dataset = PseudoLabelDataset(pseudo_ids, TEST_IMAGE_DIR, HEIGHT, WIDTH, pseudo_labels_list, confidence_masks_list)
        pseudo_loader = DataLoader(pseudo_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)

        model_scratch_pseudo = build_unet("resnet34", None, N_CLASSES).to(DEVICE)
        model_scratch_pseudo.load_state_dict(copy.deepcopy(model_scratch.state_dict()))
        optimizer_pseudo = torch.optim.AdamW(model_scratch_pseudo.parameters(), lr=PSEUDO_LABEL_LR, weight_decay=1e-4)
        scaler_pseudo = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

        model_scratch_pseudo.train()
        for epoch in range(1, PSEUDO_LABEL_EPOCHS + 1):
            running_loss, batches = 0.0, 0
            for images, labels, conf_masks in tqdm(pseudo_loader, desc=f"Pseudo-label fine-tune {epoch}/{PSEUDO_LABEL_EPOCHS}"):
                images, labels, conf_masks = images.to(DEVICE), labels.to(DEVICE), conf_masks.to(DEVICE)
                optimizer_pseudo.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                    logits = unet_forward(model_scratch_pseudo, images)
                    loss = masked_segmentation_loss(logits, labels, conf_masks)
                scaler_pseudo.scale(loss).backward()
                scaler_pseudo.step(optimizer_pseudo)
                scaler_pseudo.update()
                running_loss += float(loss.item())
                batches += 1
            print(f"epoch {epoch}/{PSEUDO_LABEL_EPOCHS} | loss {running_loss / max(batches, 1):.5f}")

        pseudo_forward_fn = TTA_FORWARD_FNS["scratch"] if USE_TTA_FOR_CALIBRATION else unet_forward
        cal_probs_pseudo = cache_probabilities("scratch_pseudo", pseudo_forward_fn, model_scratch_pseudo, cal_df, cal_loader, "cal")
        holdout_probs_pseudo = cache_probabilities("scratch_pseudo", pseudo_forward_fn, model_scratch_pseudo, holdout_df, holdout_loader, "holdout")

        pseudo_model_order = FULL_MODEL_ORDER + ["scratch_pseudo"]
        pseudo_cal_probs = dict(cal_probs)
        pseudo_cal_probs["scratch_pseudo"] = cal_probs_pseudo
        pseudo_holdout_probs = dict(holdout_probs)
        pseudo_holdout_probs["scratch_pseudo"] = holdout_probs_pseudo

        # Uniform sweep: how much weight should the pseudo-fine-tuned model get, rescaling
        # each class's existing weight vector proportionally rather than re-running the full
        # class-aware search (that already happened once above; this answers one new question).
        best_pseudo_score, best_pseudo_matrix, best_pseudo_thresholds = -1.0, None, None
        for pseudo_weight in np.arange(0.0, 0.41, 0.1):
            remaining = 1.0 - pseudo_weight
            candidate_matrix = {
                c: [w * remaining for w in CLASS_WEIGHT_MATRIX[c]] + [round(pseudo_weight, 3)]
                for c in CLASSES
            }
            score, thresholds, _ = evaluate_class_aware(candidate_matrix, pseudo_cal_probs, cal_masks, pseudo_model_order, THRESHOLD_GRID, CLASSES, HEIGHT, WIDTH)
            if score > best_pseudo_score:
                best_pseudo_score, best_pseudo_matrix, best_pseudo_thresholds = score, candidate_matrix, thresholds

        pseudo_holdout_score, _ = evaluate_fixed_class_aware(best_pseudo_matrix, best_pseudo_thresholds, pseudo_holdout_probs, holdout_masks, pseudo_model_order, CLASSES, HEIGHT, WIDTH)
        print("Pseudo-labeling holdout score:", round(pseudo_holdout_score, 6), "| baseline holdout score:", round(holdout_score, 6))

        if pseudo_holdout_score > holdout_score:
            print("Pseudo-labeling improved the holdout score — adopting it for the final submission.")
            FULL_MODEL_ORDER = pseudo_model_order
            CLASS_WEIGHT_MATRIX = best_pseudo_matrix
            FINAL_THRESHOLDS = best_pseudo_thresholds
            MODELS_BY_NAME["scratch_pseudo"] = model_scratch_pseudo
            BASE_FORWARD_FNS["scratch_pseudo"] = unet_forward
            TTA_FORWARD_FNS["scratch_pseudo"] = make_tta_forward(unet_forward)
            holdout_score = pseudo_holdout_score
        else:
            print("Pseudo-labeling did not improve the holdout score — keeping the original blend.")
else:
    print("USE_PSEUDO_LABELING is False — skipped.")

USE_PSEUDO_LABELING is False — skipped.


## Final test inference and submission

Same RLE/validation checks your own notebook already runs — row count, column order, duplicate check, empty-mask handling.

In [45]:
# No component-size filtering is applied by default: it was not part of calibration/holdout scoring.
# Keeping all predicted components avoids silently deleting small defects.
MIN_COMPONENT_AREA = 0

def remove_small_components(binary_mask, min_area):
    binary_mask = np.asarray(binary_mask, dtype=np.uint8)
    if min_area <= 1:
        return binary_mask
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    cleaned = np.zeros_like(binary_mask, dtype=np.uint8)
    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == label] = 1
    return cleaned

In [46]:
# Run the ensemble on every test image using the same inference mode used during calibration.
test_probs = predict_ensemble_on_loader(
    CLASS_WEIGHT_MATRIX, FULL_MODEL_ORDER, test_loader,
    use_tta=USE_TTA_AT_FINAL_INFERENCE
)

# Normalize prediction dictionary keys once and reject collisions, missing IDs, and extras.
# This avoids silently dropping an unexpected prediction or failing on str/Path key mismatches.
test_probs_by_id = {str(key): value for key, value in test_probs.items()}
if len(test_probs_by_id) != len(test_probs):
    raise RuntimeError("Prediction dictionary contains duplicate keys after string normalization")
expected_test_ids = [str(x) for x in TEST_IDS]
if len(expected_test_ids) != len(set(expected_test_ids)):
    raise RuntimeError("TEST_IDS contains duplicate image IDs")
missing_test_ids = sorted(set(expected_test_ids) - set(test_probs_by_id))
extra_test_ids = sorted(set(test_probs_by_id) - set(expected_test_ids))
if missing_test_ids or extra_test_ids:
    raise RuntimeError(
        f"Test prediction ID coverage mismatch: missing={len(missing_test_ids)}, "
        f"extra={len(extra_test_ids)}; missing examples={missing_test_ids[:5]}; "
        f"extra examples={extra_test_ids[:5]}"
    )

threshold_array = np.array([FINAL_THRESHOLDS[name] for name in CLASSES], dtype=np.float32)
records = []
for image_id in tqdm(expected_test_ids, desc="Encoding submission"):
    probs = np.asarray(test_probs_by_id[image_id], dtype=np.float32)
    if probs.shape != (N_CLASSES, HEIGHT, WIDTH):
        raise ValueError(f"Unexpected probability shape for {image_id}: {probs.shape}")
    if not np.isfinite(probs).all():
        raise ValueError(f"Non-finite probabilities for test image {image_id}")
    for c, class_name in enumerate(CLASSES):
        binary_mask = (probs[c] >= threshold_array[c]).astype(np.uint8)
        binary_mask = remove_small_components(binary_mask, MIN_COMPONENT_AREA)
        records.append({
            "image_id": image_id,
            "defect_type": str(class_name),
            # encode_rle returns exactly " " for an empty mask, as in the
            # successfully executed V15 notebook.
            "mask_rle": encode_rle(binary_mask),
        })

KEY_COLUMNS = ["image_id", "defect_type"]
EXPECTED_COLUMNS = ["ID", "image_id", "defect_type", "mask_rle"]

# Validate the official template before using it as the submission skeleton.
required_template_columns = ["ID", "image_id", "defect_type", "mask_rle"]
if sample_submission.columns.tolist() != required_template_columns:
    raise ValueError(f"Unexpected sample_submission columns: {sample_submission.columns.tolist()}")
if sample_submission[KEY_COLUMNS + ["ID"]].isna().any().any():
    raise ValueError("Official sample_submission contains null IDs or image/class keys")
for column in ["ID", "image_id", "defect_type"]:
    if sample_submission[column].astype(str).str.strip().eq("").any():
        raise ValueError(f"Official sample_submission contains blank values in {column}")
if sample_submission.duplicated(subset=KEY_COLUMNS).any():
    raise ValueError("Official sample_submission contains duplicate image/class keys")

predictions_df = pd.DataFrame(records, columns=KEY_COLUMNS + ["mask_rle"])
predictions_df["image_id"] = predictions_df["image_id"].astype(str)
predictions_df["defect_type"] = predictions_df["defect_type"].astype(str)

if predictions_df.empty:
    raise RuntimeError("The inference pipeline produced no prediction records")
if predictions_df[KEY_COLUMNS].isna().any().any():
    raise ValueError("Prediction output contains null image/class keys")
if predictions_df[KEY_COLUMNS].astype(str).apply(lambda col: col.str.strip().eq("")).any().any():
    raise ValueError("Prediction output contains blank image/class keys")
if predictions_df["mask_rle"].isna().any():
    raise ValueError("Prediction output contains null mask_rle values")
if not predictions_df["mask_rle"].map(lambda value: isinstance(value, str)).all():
    raise TypeError("Every predicted mask_rle value must be a string")
if predictions_df["mask_rle"].eq("").any():
    raise ValueError("Encoder emitted raw empty-string mask_rle; expected a single space for empty masks")
if predictions_df.duplicated(subset=KEY_COLUMNS).any():
    raise ValueError("Prediction output contains duplicate image/class keys")

sample_keys = sample_submission[KEY_COLUMNS].astype(str).copy()
prediction_keys = predictions_df[KEY_COLUMNS].copy()
if len(predictions_df) != len(sample_submission):
    raise ValueError(
        f"Prediction row count {len(predictions_df)} does not match sample template {len(sample_submission)}"
    )

# Outer-join coverage audit catches BOTH missing predictions and unexpected extras.
coverage = sample_keys.merge(
    prediction_keys, on=KEY_COLUMNS, how="outer", indicator=True, validate="one_to_one"
)
missing_pairs = coverage.loc[coverage["_merge"] == "left_only", KEY_COLUMNS]
extra_pairs = coverage.loc[coverage["_merge"] == "right_only", KEY_COLUMNS]
if not missing_pairs.empty or not extra_pairs.empty:
    raise RuntimeError(
        "Prediction/template key mismatch. "
        f"Missing predictions: {len(missing_pairs)}; unexpected predictions: {len(extra_pairs)}. "
        f"Missing examples: {missing_pairs.head(5).to_dict('records')}; "
        f"extra examples: {extra_pairs.head(5).to_dict('records')}"
    )

submission = sample_submission[["ID", "image_id", "defect_type"]].copy()
submission["image_id"] = submission["image_id"].astype(str)
submission["defect_type"] = submission["defect_type"].astype(str)
submission = submission.merge(
    predictions_df, on=KEY_COLUMNS, how="left", validate="one_to_one", sort=False
)

# A NaN here means the prediction did not join. Never fill it with an empty mask.
if submission["mask_rle"].isna().any():
    bad = int(submission["mask_rle"].isna().sum())
    raise ValueError(f"{bad} submission rows have null mask_rle after the validated merge")
if not submission["mask_rle"].map(lambda value: isinstance(value, str)).all():
    raise TypeError("Merged submission contains non-string mask_rle values")
if submission["mask_rle"].eq("").any():
    raise ValueError("Raw empty-string mask_rle remains; expected exactly one space for empty masks")

# Enforce exactly one space for any whitespace-only empty mask; do not alter non-empty RLE.
empty_rle = submission["mask_rle"].str.strip().eq("")
submission.loc[empty_rle, "mask_rle"] = " "

if len(submission) != len(sample_submission) or len(submission) != 12000:
    raise ValueError(f"Submission row count mismatch: {len(submission)}")
if submission.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected submission columns: {submission.columns.tolist()}")
if submission[["ID", "image_id", "defect_type"]].isna().any().any():
    raise ValueError("Null ID/image_id/defect_type values found before writing submission")
if submission.duplicated(subset=KEY_COLUMNS).any():
    raise ValueError("Duplicate image/class submission rows")
if not submission[["ID", "image_id", "defect_type"]].astype(str).reset_index(drop=True).equals(
    sample_submission[["ID", "image_id", "defect_type"]].astype(str).reset_index(drop=True)
):
    raise ValueError("Submission row order/IDs/classes differ from sample_submission.csv")

# Validate RLE pairs and bounds before writing the file.
def validate_submission_rle(value, height=HEIGHT, width=WIDTH):
    if not isinstance(value, str):
        return False
    if value.strip() == "":
        return value == " "  # only the exact single-space empty-mask sentinel is allowed
    try:
        values = np.asarray(value.split(), dtype=np.int64)
        if len(values) % 2:
            return False
        starts = values[0::2]
        lengths = values[1::2]
        if np.any(starts < 1) or np.any(lengths < 1):
            return False
        ends = starts + lengths - 1
        if np.any(ends > height * width):
            return False
        if len(starts) > 1 and np.any(starts[1:] <= ends[:-1]):
            return False
        return True
    except (ValueError, TypeError, OverflowError):
        return False

rle_valid = submission["mask_rle"].map(validate_submission_rle)
if not rle_valid.all():
    raise ValueError(f"Invalid RLE rows: {int((~rle_valid).sum())}")

# Save the exact Kaggle filename plus versioned and output-folder backups.
mode_suffix = "tta_gpu" if USE_TTA_AT_FINAL_INFERENCE else "notta_cpu"
versioned_submission_path = OUTPUT_DIR / f"submission_v9_2_5_{mode_suffix}.csv"
root_submission_path = Path("/kaggle/working/submission.csv")
output_submission_path = OUTPUT_DIR / "submission.csv"
temp_versioned = versioned_submission_path.with_name(versioned_submission_path.stem + ".tmp.csv")
temp_root = root_submission_path.with_name("submission.tmp.csv")
temp_output = output_submission_path.with_name("submission.tmp.csv")
submission.to_csv(temp_versioned, index=False, quoting=csv.QUOTE_ALL)
submission.to_csv(temp_root, index=False, quoting=csv.QUOTE_ALL)
submission.to_csv(temp_output, index=False, quoting=csv.QUOTE_ALL)
os.replace(temp_versioned, versioned_submission_path)
os.replace(temp_root, root_submission_path)
os.replace(temp_output, output_submission_path)

# Reload the actual CSV with NA parsing disabled so the one-space sentinel remains a string.
read_options = {
    "dtype": {"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    "keep_default_na": False,
}
final_check = pd.read_csv(root_submission_path, **read_options)
if final_check.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Saved submission columns mismatch: {final_check.columns.tolist()}")
if final_check.shape != (12000, 4):
    raise ValueError(f"Saved submission shape mismatch: {final_check.shape}")
if final_check.isna().any().any():
    raise ValueError("Null values found in saved submission.csv")
if final_check[["ID", "image_id", "defect_type"]].apply(lambda col: col.str.strip().eq("")).any().any():
    raise ValueError("Blank ID/image_id/defect_type values found in saved submission.csv")
if final_check["mask_rle"].eq("").any():
    raise ValueError("Raw empty-string mask_rle values remain; expected one space for empty masks")
empty_mask_rows = final_check["mask_rle"].str.strip().eq("")
if (final_check.loc[empty_mask_rows, "mask_rle"] != " ").any():
    raise ValueError("Empty masks must be encoded as exactly one space")
if not final_check["mask_rle"].map(validate_submission_rle).all():
    raise ValueError("Saved submission contains invalid RLE masks")
for col in ["ID", "image_id", "defect_type"]:
    if final_check[col].tolist() != sample_submission[col].astype(str).tolist():
        raise ValueError(f"Saved submission {col} order differs from sample_submission.csv")
if final_check.duplicated(subset=KEY_COLUMNS).any():
    raise ValueError("Duplicate image/class rows in saved submission.csv")
class_counts = final_check["defect_type"].value_counts().reindex(CLASSES, fill_value=0)
if not (class_counts == 3000).all():
    raise ValueError(f"Unexpected per-class row counts: {class_counts.to_dict()}")

print("=" * 68)
print("FINAL SUBMISSION VALIDATION: PASSED")
print("=" * 68)
print("Prediction/template pairs matched exactly:", len(coverage))
print("Null prediction RLE values before writing:", int(predictions_df["mask_rle"].isna().sum()))
print("Null submission values after CSV round-trip:", int(final_check.isna().sum().sum()))
print("Empty masks (single-space RLE):", int(empty_mask_rows.sum()))
print("Non-empty masks:", int((~empty_mask_rows).sum()))
print("Kaggle submission file:", root_submission_path)
print("Kaggle output-folder copy:", output_submission_path)
print("Versioned backup:", versioned_submission_path)
print("Rows:", len(final_check), "| Unique test images:", final_check["image_id"].nunique())
print("Class counts:", class_counts.to_dict())
print("File size (MB):", round(root_submission_path.stat().st_size / (1024 ** 2), 3))
print("Execution device:", DEVICE, "| TTA:", USE_TTA_AT_FINAL_INFERENCE)
print("Submission CSV is ready for Kaggle.")
final_check.head(10)


Ensemble inference (TTA=True):   0%|          | 0/3000 [00:00<?, ?it/s]

Encoding submission:   0%|          | 0/3000 [00:00<?, ?it/s]

FINAL SUBMISSION VALIDATION: PASSED
Prediction/template pairs matched exactly: 12000
Null prediction RLE values before writing: 0
Null submission values after CSV round-trip: 0
Empty masks (single-space RLE): 7384
Non-empty masks: 4616
Kaggle submission file: /kaggle/working/submission.csv
Kaggle output-folder copy: /kaggle/working/surfaceguard_v9_2_5_outputs/submission.csv
Versioned backup: /kaggle/working/surfaceguard_v9_2_5_outputs/submission_v9_2_5_tta_gpu.csv
Rows: 12000 | Unique test images: 3000
Class counts: {'defect_a': 3000, 'defect_b': 3000, 'defect_c': 3000, 'defect_d': 3000}
File size (MB): 2.751
Execution device: cuda:0 | TTA: True
Submission CSV is ready for Kaggle.


,ID,image_id,defect_type,mask_rle
0,1,WAFNKCRR7Q,defect_a,
1,2,WAFNKCRR7Q,defect_b,
2,3,WAFNKCRR7Q,defect_c,100666 1
3,4,WAFNKCRR7Q,defect_d,
4,5,YZXKEWSUDD,defect_a,
5,6,YZXKEWSUDD,defect_b,
6,7,YZXKEWSUDD,defect_c,
7,8,YZXKEWSUDD,defect_d,
8,9,0ATN06ZB48,defect_a,62543 5 62612 20 62669 10 62706 5 62735 31 627...
9,10,0ATN06ZB48,defect_b,63005 2 63056 6 63130 15 63180 13 63258 22 633...


## After submission generation and validation

The previous code cell writes:
- `/kaggle/working/submission.csv` — the exact competition submission filename.
- `/kaggle/working/surfaceguard_v9_2_5_outputs/submission.csv` — a second copy.
- A versioned CSV indicating whether GPU TTA or CPU single-pass inference was used.

The final validation cell independently reloads the CSV and checks nulls, exact single-space empty-mask representation, row order, image/class key coverage, duplicate keys, RLE validity, 12,000 rows, and 3,000 rows per class.

To send it to the leaderboard, use Kaggle's notebook **Save Version** workflow and submit the generated `submission.csv` from the notebook output. This notebook does not invoke the Kaggle CLI or need a Kaggle API token.


In [47]:
# FINAL CELL — strict revalidation of the Kaggle submission artifact.
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
EXPECTED_COLUMNS = ["ID", "image_id", "defect_type", "mask_rle"]
KEY_COLUMNS = ["image_id", "defect_type"]

if not SUBMISSION_PATH.is_file():
    raise FileNotFoundError(f"Missing validated submission: {SUBMISSION_PATH}")
if not SAMPLE_SUBMISSION_PATH.is_file():
    raise FileNotFoundError(f"Missing official sample template: {SAMPLE_SUBMISSION_PATH}")

read_opts = {
    "dtype": {"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    "keep_default_na": False,
}
sample_check = pd.read_csv(SAMPLE_SUBMISSION_PATH, **read_opts)
submission_check = pd.read_csv(SUBMISSION_PATH, **read_opts)

if sample_check.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected sample template columns: {sample_check.columns.tolist()}")
if submission_check.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected submission columns: {submission_check.columns.tolist()}")
if submission_check.shape != sample_check.shape or submission_check.shape != (12000, 4):
    raise ValueError(f"Submission shape {submission_check.shape} does not match official template {sample_check.shape}")
if sample_check.isna().any().any():
    raise ValueError("Null values found in official sample template")
if submission_check.isna().any().any():
    raise ValueError("Null values found in submission")
for column in ("ID", "image_id", "defect_type"):
    if submission_check[column].str.strip().eq("").any():
        raise ValueError(f"Blank values found in required column {column}")
    if submission_check[column].tolist() != sample_check[column].tolist():
        raise ValueError(f"Submission {column} sequence differs from the official template")
if submission_check["mask_rle"].eq("").any():
    raise ValueError("Raw empty-string mask_rle fields found; empty masks must be exactly one space")
empty_mask_rows = submission_check["mask_rle"].str.strip().eq("")
if (submission_check.loc[empty_mask_rows, "mask_rle"] != " ").any():
    raise ValueError("Empty RLE fields must use exactly one space, not null or arbitrary whitespace")
if submission_check.duplicated().any():
    raise ValueError("Duplicate submission rows found")
if submission_check.duplicated(subset=KEY_COLUMNS).any():
    raise ValueError("Duplicate image/class rows found")
if not submission_check["mask_rle"].map(lambda value: validate_submission_rle(value, HEIGHT, WIDTH)).all():
    raise ValueError("Invalid RLE value found in submission")

class_counts = submission_check["defect_type"].value_counts().reindex(CLASSES, fill_value=0)
if not (class_counts == 3000).all():
    raise ValueError(f"Unexpected per-class row counts: {class_counts.to_dict()}")

print("\nFINAL KAGGLE SUBMISSION VALIDATION: PASSED")
print("Null values:", int(submission_check.isna().sum().sum()))
print("Empty masks represented by exactly one space:", int(empty_mask_rows.sum()))
print("Non-empty masks:", int((~empty_mask_rows).sum()))
print("Competition:", COMPETITION)
print("Rows:", len(submission_check), "| Test images:", submission_check["image_id"].nunique())
print("Holdout diagnostic:", round(float(holdout_score), 6))
print("Device:", DEVICE, "| TTA:", USE_TTA_AT_FINAL_INFERENCE)
print("Submission file:", SUBMISSION_PATH)
print("File size (MB):", round(SUBMISSION_PATH.stat().st_size / (1024 ** 2), 3))
print("\nThe validated submission.csv is ready in Kaggle notebook output.")
print("Save a notebook version, then submit the generated CSV to the competition.")



FINAL KAGGLE SUBMISSION VALIDATION: PASSED
Null values: 0
Empty masks represented by exactly one space: 7384
Non-empty masks: 4616
Competition: industrial-surface-defect-detection
Rows: 12000 | Test images: 3000
Holdout diagnostic: 0.725936
Device: cuda:0 | TTA: True
Submission file: /kaggle/working/submission.csv
File size (MB): 2.751

The validated submission.csv is ready in Kaggle notebook output.
Save a notebook version, then submit the generated CSV to the competition.


## Interpreting and submitting the result

- `FINAL KAGGLE SUBMISSION VALIDATION: PASSED` means the file passed the notebook's structural, null, coverage, ordering, class-count, and RLE checks. It does not mean Kaggle has evaluated its score.
- The local holdout score is diagnostic; it is not a guarantee or numerical forecast of the public leaderboard score.
- Compare the processed public score against the previous best **0.63881** on the competition Submissions page.
- A blank predicted mask is represented as exactly one space in `mask_rle`; it is valid and is not a null value. Missing predictions and null RLE values stop submission generation.
- Keep the V13 **0.63881** submission and your V14 four-model notebook unchanged until the new Kaggle score is available.
